# La publicité fait-elle vendre ?

Ce test proposé par Numberly consiste en 2 étapes :

1) Comprendre le marché ;
2) Comprendre l’impact de la publicité sur les ventes.


**Convention de nommage des tables :**

| Niveau | Noms | Contenu | Sert à |
|---|---|---|---|
| A | `retailer`, `programmatic_pub`, `tv_pub`, `socio_demo`, `mapping` | Brut, jamais modifié | Audit uniquement |
| B | `retailer_c`, `prog_c`, `tv_c`, `socio_c`, `mapping_c` | Doublons retirés, types convertis, "unknown" remplacés par NaN | Totaux : CA, budget média, funnel, saisonnalité |
| C | `retailer_id`, `prog_id`, `tv_id`, `socio_id`, `mapping_id` | Niveau B sans les lignes dont l'identifiant est manquant | Tout ce qui passe par un client : jointures, attribution, panier moyen par acheteur |



## 1 ) Chargement et préparation des données

### 1.1 ) Chargement

**Bibliothèques nécessaires :**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

**Chargement des données :**

In [ ]:
retailer = pd.read_csv("retailer.csv")
programmatic_pub = pd.read_csv("programmatic_publisher.csv")
tv_pub = pd.read_csv("tv_publisher.csv")
socio_demo = pd.read_csv("socio_demo.csv")
mapping = pd.read_csv("mapping_transac_publisher_tv.csv")

datasets = {
    "retailer": retailer,
    "programmatic_pub": programmatic_pub,
    "tv_pub": tv_pub,
    "socio_demo": socio_demo,
    "mapping": mapping,
}

print("Données chargées :")
for name, df in datasets.items():
    print(f"  {name:<18}: {df.shape}")

### 1.2 ) Audit des données

Toutes les vérifications de cette section portent sur les tables brutes (niveau A), sans aucune modification.

#### 1.2.1 ) Types des variables :

In [ ]:
for name, df in datasets.items():
    print(f"\n{'=' * 50}\n{name}\n{'=' * 50}")
    df.info()

**Point de vigilance :**

- les types de certaines variables ne sont pas corrects (ex : les variables de temps).

#### 1.2.2 ) Statistiques de base :

In [ ]:
for name, df in datasets.items():
    print(f"\n{'=' * 50}\n{name}\n{'=' * 50}")
    print(df.describe(include="all").round(2))

**Points de vigilance :**

- les identifiants (`customer_id`, `dsp_id`, `device_id`) contiennent tous la valeur "unknown", qui équivaut à une valeur manquante;
- les variables `sales` et `quantity` de la table retailer ont comme valeur minimale 0.

#### 1.2.3 ) Valeurs manquantes et atypiques :

##### Valeurs manquantes (NaN) :

In [ ]:
for name, df in datasets.items():
    missing = pd.DataFrame({
        "Valeurs manquantes": df.isna().sum(),
        "Pourcentage (%)": (df.isna().mean() * 100).round(0).astype(int)
    })
    print(f"\n{'=' * 50}\n{name}\n{'=' * 50}")
    print(missing)

**Récapitulatif :**

- retailer : valeurs manquantes dans brand, product_name, sales et quantity.
- mapping : valeurs manquantes dans device_id.
- Les autres tables ne présentent aucune valeur manquante.

**Zoom sur la table `retailer`**

Chaque ligne correspond à un événement enregistré sur le site du distributeur. Il faut distinguer les informations manquantes selon la nature de l'événement.

In [ ]:
retailer.sample(5)

In [ ]:
pd.DataFrame({
    "Lignes": retailer["event_name"].value_counts(),
    "Part (%)": (retailer["event_name"].value_counts(normalize=True) * 100).round(0).astype(int)
})


Trois étapes successives existent :

- Product Page View : Le client consulte une fiche produit 
- Add to cart       : Il ajoute le produit au panier       
- Order             : Il achète                            

***Variables `sales` et `quantity` :***

On ne peut parler de montant et de quantité que s'il y a eu une vente : les valeurs manquantes de `sales` et `quantity` semblent être structurelles. Vérifions que les valeurs manquantes correspondent bien aux événements sans achat.

In [ ]:
missing_sales_qt = retailer[retailer[["sales", "quantity"]].isna().any(axis=1)]
print("Événements associés aux valeurs manquantes de sales / quantity :")
print(missing_sales_qt["event_name"].value_counts(dropna=False))

**Récapitulatif :**

- les valeurs manquantes concernent uniquement les événements `Product Page View` et `Add to cart`. Aucun événement `Order` n'est concerné.

**Décision :** aucune imputation : les valeurs manquantes sont structurelles. Les indicateurs de vente sont calculés uniquement sur les événements `Order`. 

***Variables `brand` et `product_name`***

In [ ]:
missing_brand = retailer[retailer["brand"].isna()]
print("Brand et product_name manquants sur les mêmes lignes :",
      missing_brand.equals(retailer[retailer["product_name"].isna()]))


Les valeurs manquantes concernent simultanément les deux variables : quand l’une manque, l’autre manque également.

In [ ]:
# Répartition par type d'événement
events = pd.DataFrame({
    "Valeurs manquantes": missing_brand["event_name"].value_counts(),
    "Total": retailer["event_name"].value_counts()
})
events["Part (%)"] = (events["Valeurs manquantes"] / events["Total"] * 100).round(1)
print(events)

# Poids dans les ventes
orders = retailer[retailer['event_name'] == 'Order']
poids_na = orders.groupby(orders['brand'].isna()).agg(
    count=('sales', 'count'),
    CA=('sales', 'sum'),
    CA_moyen=('sales', 'mean'),
    quantite=('quantity', 'sum')
)
poids_na["Prix unitaire moyen"] = (poids_na["CA"] / poids_na["quantite"]).round(2)
poids_na["Part du CA (%)"] = (poids_na["CA"] / poids_na["CA"].sum() * 100).round(2)
print(f"\n{poids_na.drop(columns='Part du CA (%)').round(0).astype(int).join(poids_na["Part du CA (%)"])}")

# Répartition temporelle
mois_total = retailer["timestamp_utc"].str[:7].value_counts().sort_index()
mois_manq = missing_brand["timestamp_utc"].str[:7].value_counts().sort_index()
print(f"\n{pd.DataFrame({'manquants': mois_manq, 'part (%)': (mois_manq / mois_total * 100).round(2)})}")

# Clients concernés
print("\nClients concernés :", missing_brand["customer_id"].nunique(),
      "sur", retailer["customer_id"].nunique(), "clients distincts (", round(missing_brand["customer_id"].nunique()/retailer["customer_id"].nunique()*100,2),"%)" )

**Récapitulatif :**

- Les manquants sont présents sur les trois types d'événements, mais restent minoritaires dans chacun.
- Ils sont répartis sur toute la période étudiée (janvier à juin 2024).
- Ils concernent 10% des clients
- Les 8 776 lignes de commandes concernées représentent 699 706 \$, soit 1.12 % du CA total. Leur prix unitaire moyen par ligne est supérieur à celui des autres commandes (74 \$ contre 40 \$).

L'absence simultanée de marque et de produit suggère une information produit non disponible à la source. Ces lignes ne sont pas concentrées sur un mois ou quelques clients.

**Décision :** Aucune imputation. Les lignes sont exclues des analyses par marque et par produit, mais conservées pour les analyses globales de ventes et au niveau client.

**Zoom sur la table `mapping`**


In [ ]:
nan_dev = mapping[mapping["device_id"].isna()]
nan_dev.head()

Les valeurs manquantes de `device_id` semblent concerner les lignes où `customer_id` est inconnu. Vérifions si c'est le seul cas :

In [ ]:
print("customer_id des lignes avec device_id manquant :")
print(nan_dev["customer_id"].value_counts(dropna=False))

unknown_cust = mapping[mapping["customer_id"] == "unknown"]
print("\ndevice_id des lignes avec customer_id unknown :")
print(unknown_cust["device_id"].value_counts(dropna=False))

Toutes les valeurs manquantes de `device_id` correspondent à un `customer_id` égal à "unknown", et inversement. 

**Décision :** Ces lignes ne permettent pas de relier un client à un identifiant TV. Elles sont inutilisables pour l'attribution publicitaire.

##### Valeurs "Unknown" :

Les identifiants inconnus sont parfois codés sous forme de chaîne de caractères ("unknown") plutôt que de valeurs NaN. Ils doivent donc être traités séparément. La variable `device_type` contient également des valeurs "Unknown".

In [ ]:
def count_unknown(datasets):
    id_cols = ["customer_id", "dsp_id", "device_id","device_type"]
    results = []
    for name, df in datasets.items():
        for col in id_cols:
            if col in df.columns:
                n_unknown = df[col].astype("string").str.strip().str.lower().eq("unknown").sum()

                results.append({
                    "Table": name,
                    "Variable": col,
                    "Nombre de unknown": n_unknown,
                    "Pourcentage (%)": round(n_unknown / len(df) * 100,2) if len(df) > 0 else 0
                })
    return pd.DataFrame(results)

unknown_summary = count_unknown(datasets)
print(unknown_summary)

Pour la table `mapping`, une ligne sert à relier un client à ses identifiants média. Elle est donc utile seulement si le customer_id est connu :

- customer_id unknown : ligne inutilisable pour relier une pub à un achat. On l’exclut.
- dsp_id unknown (client connu) : exposition programmatique non mesurable.
- device_id unknown (client connu) : exposition TV non mesurable.

Pour les autres tables, un identifiant "unknown" ne représente pas un individu identifiable. Il ne doit donc pas être utilisé comme clé de jointure : plusieurs lignes seraient artificiellement associées au même identifiant.  

**Récapitulatif :**

- Les lignes de `mapping` pour lesquelles le customer_id est "unknown" sont supprimées.
- Pour les identifiants dans les autres tables :
    - Les lignes sont conservées pour les totaux globaux : les dépenses publicitaires et les ventes ont bien eu lieu.
    - Elles sont exclues des analyses nécessitant une identification individuelle.
- Les expositions non mesurables ne sont pas assimilées à des non-expositions.
- `device_type` n’est pas un identifiant mais une caractéristique de l’impression (type d’écran). Ses 686 valeurs "unknown" (0,004 % des impressions) sont des valeurs manquantes. Elles sont remplacées par NaN, mais les lignes sont conservées : l’impression a bien été diffusée et payée.

##### Valeurs nulles dans `sales` et `quantity` :

Contrairement aux NaN, les valeurs nulles sont des valeurs renseignées. Il faut donc vérifier si elles correspondent à de véritables transactions ou à des anomalies.

**Identification des lignes de commandes concernées**

In [ ]:
print(f"sales == 0            : {(orders['sales'] == 0).sum()}")
print(f"quantity == 0         : {(orders['quantity'] == 0).sum()}")
nulle = orders[(orders["sales"]==0) | (orders["quantity"]==0)].copy()
print(f"Part des lignes Order : {len(nulle) /len(orders):.2%}")

# Répartition croisée des cas selon sales et quantity
cas = pd.crosstab(nulle["sales"] ==0, nulle["quantity"] ==0,rownames=["sales == 0"], colnames=["quantity == 0"])
cas = cas.reindex(
    index=[False, True],
    columns=[False, True],
    fill_value=0
)
print(f"\n{cas}")

**Récapitulatif :**

- Les lignes concernées représentent moins de 1% des lignes de commandes.
- Toutes les lignes avec une quantité nulle ont également un montant nul.

On distingue donc 2 situations :

- Les lignes sans montant ni quantité : 2 767 lignes.
- Les lignes sans montant mais avec une quantité positive : 3 638 lignes.

***Lignes sans montant ni quantité***

Ces lignes ne contiennent aucune information commerciale.

**Décision :** Suppression de ces lignes car elles ne correspondent à aucune vente exploitable.

***Lignes sans montant mais avec une quantité positive***

Ces 3 638 lignes peuvent correspondre à des produits offerts. Il faut vérifier si elles présentent un comportement différent des achats classiques.

In [ ]:
# Comparaison des quantités
offerts = orders[(orders["sales"] == 0) & (orders["quantity"] > 0)].copy()
normales = orders[(orders["sales"] > 0) & (orders["quantity"] > 0)].copy()
print(pd.DataFrame({
    "Sans montant": offerts["quantity"].describe(),
    "Achats classiques": normales["quantity"].describe()
}).round(2))

# Produits et marques concernés
poids = pd.DataFrame({
    "Sans montant (%)": offerts["product_name"].value_counts(normalize=True) * 100,
    "Achats classiques (%)": normales["product_name"].value_counts(normalize=True) * 100,
}).sort_values("Sans montant (%)", ascending=False).head(10).round(2)
print(poids)
print("\nRépartition par marque :")
print(offerts["brand"].value_counts(dropna=False, normalize=True).round(3))

# Répartition chez les clients
print("\nClients distincts :", offerts["customer_id"].nunique(), "pour", len(offerts), "lignes")
print("Part des clients concernés parmi les acheteurs :",
      f"{offerts['customer_id'].nunique() / orders['customer_id'].nunique():.2%}")
# Distribution du nombre de lignes concernées par client
print("\nDistribution du nombre de lignes par client :")
distribution = offerts.groupby("customer_id").size().value_counts().sort_index()
print(pd.DataFrame({
    "Nombre de clients": distribution,
    "Part (%)": (distribution / distribution.sum() * 100).round(2)
}))

# Répartition dans le temps
offerts["timestamp_utc"] = pd.to_datetime(offerts["timestamp_utc"])
print("\nRépartition mensuelle :")
print(offerts["timestamp_utc"].dt.to_period("M").value_counts().sort_index())

**Récapitulatif :**

- Ces lignes comportent presque toujours une seule unité.
- Elles ne sont pas concentrées sur un produit ou une marque particulière.
- Elles sont présentes sur toute la période et concernent plusieurs milliers de clients.

Elles peuvent correspondre à des produits offerts ou à des erreurs d'enregistrement du montant associé à la commande.

**Décision :** Suppression de ces lignes : elles ne génèrent aucun chiffre d'affaires et représentent une faible part des lignes de commande (moins de 1 %).

#### 1.2.4 ) Doublons :

In [ ]:
for name, df in datasets.items():
    print(f"{name:<18}: {df.duplicated().sum():>8,} doublons exacts")

On regarde s'il s'agit de vrais doublons ou de lignes volontairement répétées.

##### `retailer`

In [ ]:
dup = retailer[retailer.duplicated(keep=False)]
print(dup.sort_values(by=retailer.columns.tolist()).head(10))

print("\nLignes impliquées par type d'événement :")
print(dup["event_name"].value_counts())

##### `programmatic_pub`

In [ ]:
programmatic_pub[
    programmatic_pub.duplicated(keep=False)
].sort_values(by=programmatic_pub.columns.tolist()).head(10)

##### `tv_pub`

In [ ]:
tv_pub[
    tv_pub.duplicated(keep=False)
].sort_values(by=tv_pub.columns.tolist()).head(10)

##### `mapping`

In [ ]:
mapping[mapping.duplicated(keep=False)].head()

In [ ]:
mask_vide = (
    mapping["customer_id"].eq("unknown")
    & mapping["dsp_id"].eq("unknown")
    & mapping["device_id"].isna()
)
print("Lignes 'customer unknown + dsp unknown + device NaN' :", mask_vide.sum())
print("Doublons restants hors ces lignes :", mapping[~mask_vide].duplicated().sum())

**Décisions :**

- Pour `retailer`, `programmatic_pub` et `tv_pub`, les doublons sont exacts sur toutes les variables, avec un timestamp identique à la seconde : ils semblent involontaires : on les supprime.
- Pour `mapping`, les seuls doublons sont les lignes sans aucune information utile (`customer_id` et `dsp_id` unknown, `device_id` NaN) : on les supprime toutes.

#### Synthèse de l'audit

Les constats portent sur les tables brutes (niveau A) ; les décisions sont appliquées dans la section suivante.

Les données sont globalement exploitables : aucune imputation n'est nécessaire et `socio_demo` ne présente aucune anomalie (une ligne par client, une seule ligne `"unknown"`). Hors valeurs manquantes structurelles, chaque anomalie touche moins de 3 % des lignes, sauf l'identifiant programmatique : `dsp_id` est inconnu pour 10,9 % des impressions et 10,9 % des lignes de `mapping`.

Trois règles guident les décisions :

- une valeur manquante n'est pas forcément une anomalie : `sales` et `quantity` n'existent que pour les achats ;
- une ligne sans identifiant compte dans les totaux (la vente ou la dépense a bien eu lieu), mais pas dans les analyses par client : c'est la distinction entre les niveaux B et C ;
- une exposition non mesurable n'est pas assimilée à une non-exposition.

| Constat | Ampleur | Décision | Niveau |
|---|---|---|---|
| **Types** : dates stockées en texte, coûts en milli-cents, âge et revenu en tranches textuelles | Toutes les tables sauf `mapping` | Dates converties, coût converti en dollars, variables qualitatives en catégories (âge et revenu ordonnés) | B |
| **`sales` / `quantity` manquants** : absents de toutes les vues produit et de tous les ajouts au panier, présents sur toutes les commandes. Le manque est structurel | 85 % des lignes de `retailer` | Aucune imputation ; indicateurs de vente calculés sur les seuls `Order` | Analyse |
| **`brand` / `product_name` manquants** : toujours absents ensemble, sur les trois types d'événements et tout le semestre, sans concentration sur quelques clients | 2,8 % des lignes ; 8 776 lignes de commandes, soit 1,1 % du CA ; 10 % des clients | Aucune imputation ; lignes exclues des analyses par marque ou par produit | Analyse |
| **`mapping` sans client** : `device_id` est NaN si et seulement si `customer_id` vaut `"unknown"` ; la ligne ne relie aucun achat à une exposition | 158 701 lignes (2,0 %) | Retirées : inutilisables pour l'attribution | C |
| **Identifiants `"unknown"`** : codés en texte, donc invisibles pour `isna()` ; ils ne désignent aucun individu et fausseraient les jointures | `dsp_id` 10,9 % (programmatique), `device_id` 3,0 % (TV), `customer_id` 2,0 % (`retailer`) | Remplacés par NaN ; lignes conservées pour les totaux, retirées pour les analyses par client | B, puis C |
| **`device_type` "unknown"** (type d’écran non renseigné) | 686 impressions (< 0,01 %) | Remplacé par NaN, lignes conservées | B.
| **Client connu, canal inconnu** (`mapping`) : `dsp_id` ou `device_id` vaut `"unknown"` | ≈ 11 % des clients connus en programmatique, ≈ 3 % en TV | Lignes conservées ; canal marqué « non mesurable » | C |
| **`Order` à montant nul** : une quantité nulle implique toujours un montant nul. 2 767 lignes sans montant ni quantité ; 3 638 lignes sans montant mais avec une quantité positive (produits offerts ou erreurs de saisie), sans concentration sur un produit, une marque ou une période | 6 405 lignes, 0,45 % des commandes | Supprimées : elles ne portent aucun chiffre d'affaires | B |
| **Doublons exacts** : identiques sur toutes les variables, horodatage à la seconde compris, donc involontaires. Dans `mapping`, ils ne concernent que des lignes vides (`customer_id` et `dsp_id` à `"unknown"`, `device_id` NaN) | `retailer` 770, `programmatic_pub` 597, `tv_pub` 4 775, `mapping` 17 490 | Supprimés ; dans `mapping`, les 17 491 lignes vides sont toutes retirées | B |

### 1.3 ) Préparation des tables de données

#### 1.3.1 ) Niveau B : tables nettoyées

On commence par retirer les doublons.

In [ ]:
def drop_dups(df, name):
    out = df.drop_duplicates(ignore_index=True)
    print(f"{name:<18}: {len(df) - len(out):,} doublons retirés")
    return out

# mapping : on retire d'abord les lignes entièrement vides (customer et dsp unknown, device NaN)
mapping_c = mapping.loc[~mask_vide]
print(f"{'mapping':<18}: {mask_vide.sum():,} lignes entièrement vides retirées")

retailer_c = drop_dups(retailer, "retailer")
prog_c = drop_dups(programmatic_pub, "programmatic_pub")
tv_c = drop_dups(tv_pub, "tv_pub")
mapping_c = drop_dups(mapping_c, "mapping")
socio_c = socio_demo.copy()

Ensuite on convertit les types de variable et on ordonne les modalités d'âge et de revenu.

In [ ]:
# Types : dates, coût converti (1 milli-cent = 1/100 000 de l'unité monétaire), catégories
for df in (retailer_c, prog_c, tv_c):
    df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"])

for df in (prog_c, tv_c):
    df["cost"] = df["cost_milli_cent"] / 100_000    # conversion en dollars

for df, cols in [(retailer_c, ["event_name", "brand"]),
                 (prog_c, ["campaign_name", "device_type"]),
                 (socio_c, ["breed"])]:
    df[cols] = df[cols].astype("category")

ordre_age = ["[18-25[", "[25-35[", "[35-45[", "[45-55[", "[55-65[", "[65+["]
ordre_revenu = ["[0-40K$[", "[40-80K$[", "[80-120K$[", "[120-200K$[", "[200K$+["]
socio_c["age"] = pd.Categorical(socio_c["age"], categories=ordre_age, ordered=True)
socio_c["income"] = pd.Categorical(socio_c["income"], categories=ordre_revenu, ordered=True)
assert socio_c["age"].notna().all() and socio_c["income"].notna().all()   # aucune modalité oubliée

Puis on remplace les "unknown" par NaN :

In [ ]:
for df, cols in [(retailer_c, ["customer_id"]),
                 (prog_c, ["dsp_id"]),
                 (tv_c, ["device_id"]),
                 (socio_c, ["customer_id"]),
                 (mapping_c, ["customer_id", "dsp_id", "device_id"])]:
    for c in cols:
        df[c] = df[c].mask(df[c] == "unknown")

# device_type est déjà en catégorie : on retire la modalité "unknown", ses valeurs deviennent NaN
prog_c["device_type"] = prog_c["device_type"].cat.remove_categories("Unknown")

Pour finir on retire les lignes des Order à montant nul (quantity == 0 implique sales == 0, une seule règle suffit) :

In [ ]:
masque_zero = (retailer_c["event_name"] == "Order") & (retailer_c["sales"] == 0)
print(f"retailer : {masque_zero.sum():,} lignes Order à montant nul retirées")
retailer_c = retailer_c.loc[~masque_zero]

#### 1.3.2 ) Niveau C : tables identifiées (pour les analyses par client)

On retire les lignes dont l'identifiant est manquant. 
Dans `mapping`, on ne retire que les lignes sans `customer_id` : un client connu avec `dsp_id` ou `device_id` manquant est conservé.

In [ ]:
retailer_id = retailer_c.dropna(subset=["customer_id"])
prog_id = prog_c.dropna(subset=["dsp_id"])
tv_id = tv_c.dropna(subset=["device_id"])
socio_id = socio_c.dropna(subset=["customer_id"])

mapping_id = mapping_c.dropna(subset=["customer_id"]).copy()
mapping_id["tv_mesurable"] = mapping_id["device_id"].notna()
mapping_id["dsp_mesurable"] = mapping_id["dsp_id"].notna()

print("Part de clients mesurables :")
print(mapping_id[["tv_mesurable", "dsp_mesurable"]].mean().round(3))

#### 1.3.3 ) Contrôle des volumes, périodes et identifiants

**Volumes.** Nombre de lignes à chaque niveau, pour observer ce qui a été retiré.

In [ ]:
controle = pd.DataFrame({
    "A - brut": [len(retailer), len(programmatic_pub), len(tv_pub), len(socio_demo), len(mapping)],
    "B - nettoyé": [len(retailer_c), len(prog_c), len(tv_c), len(socio_c), len(mapping_c)],
    "C - identifié": [len(retailer_id), len(prog_id), len(tv_id), len(socio_id), len(mapping_id)],
}, index=["retailer", "programmatic_pub", "tv_pub", "socio_demo", "mapping"])

controle["Retiré A -> C (%)"] = ((1 - controle["C - identifié"] / controle["A - brut"]) * 100).round(2)
controle

**Périodes.** Les ventes et les médias doivent couvrir la même période.

In [ ]:
for name, df in [("retailer", retailer_c), ("programmatic_pub", prog_c), ("tv_pub", tv_c)]:
    print(f"{name:<18}: {df['timestamp_utc'].min()}  ->  {df['timestamp_utc'].max()}")

# Début de la TV (1re impression) : coupure avant / après de la partie 3, et durée de chaque période
debut_tv = tv_c["timestamp_utc"].min()
jours_avant = (debut_tv - pd.Timestamp("2024-01-01")) / pd.Timedelta(days=1)
jours_apres = (pd.Timestamp("2024-07-01") - debut_tv) / pd.Timedelta(days=1)
print(f"\nDébut de la TV : {debut_tv} | {jours_avant:.1f} jours avant, {jours_apres:.1f} jours après")

**Récapitulatif :**

- `retailer` et `programmatic_pub` couvrent tout le semestre, du 1er janvier au 30 juin 2024. 
- La TV ne commence que le 29 février au soir. Il y a donc une période avant (59,9 jours) et une période après (122,1 jours) son lancement.

**Identifiants.** Toutes les jointures de la suite passent par `customer_id` (ventes, profil, `mapping`), puis par `dsp_id` et `device_id` (médias). On vérifie que ces clés sont uniques là où elles doivent l'être, et que chaque client du site se retrouve dans les autres tables, et quelle part des publicités peut être rattachée à une personne.

In [ ]:
clients_site = pd.Index(retailer_id["customer_id"].unique())
clients_socio = pd.Index(socio_id["customer_id"])
clients_mapping = pd.Index(mapping_id["customer_id"])

identifiants = pd.DataFrame({
    "Identifiants distincts": [len(clients_site), clients_socio.nunique(), clients_mapping.nunique()],
    "Dont clients du site": [len(clients_site), clients_socio.isin(clients_site).sum(),
                             clients_mapping.isin(clients_site).sum()],
}, index=["retailer", "socio_demo", "mapping"])
identifiants["Part de clients du site (%)"] = (identifiants["Dont clients du site"]
                                               / identifiants["Identifiants distincts"] * 100)
display(identifiants.style.format(precision=1, thousands=" ", decimal=","))

# Clés uniques : une ligne par personne dans socio_demo et mapping, un dsp_id et un device_id par personne au plus
assert clients_socio.is_unique and clients_mapping.is_unique
assert mapping_id["dsp_id"].dropna().is_unique and mapping_id["device_id"].dropna().is_unique
# Tous les clients du site ont un profil et une ligne dans mapping
assert clients_site.isin(clients_socio).all() and clients_site.isin(clients_mapping).all()
print("Clés uniques, clients du site tous présents dans socio_demo et mapping : OK")

In [ ]:
# Publicités rattachables à une personne : identifiant renseigné (niveau C) et présent dans mapping
rattachement = pd.DataFrame({
    "Impressions (niveau B)": [len(prog_c), len(tv_c)],
    "Avec identifiant (niveau C)": [len(prog_id), len(tv_id)],
    "Identifiant présent dans mapping": [prog_id["dsp_id"].isin(mapping_id["dsp_id"]).sum(),
                                         tv_id["device_id"].isin(mapping_id["device_id"]).sum()],
    "Identifiants uniques absents de mapping": [
        prog_id.loc[~prog_id["dsp_id"].isin(mapping_id["dsp_id"]), "dsp_id"].nunique(),
        tv_id.loc[~tv_id["device_id"].isin(mapping_id["device_id"]), "device_id"].nunique()
    ]
}, index=["Programmatique", "TV"])
rattachement["Rattachables (% du niveau B)"] = (rattachement["Identifiant présent dans mapping"]
                                               / rattachement["Impressions (niveau B)"] * 100)
display(rattachement.style.format(precision=1, thousands=" ", decimal=","))

**Récapitulatif :**

- Clés fiables : une seule ligne par personne dans `socio_demo` et `mapping`, et chaque `dsp_id` ou `device_id` n'appartient qu'à une personne. Les jointures ne dupliquent donc aucune ligne.
-   Clients : tous les clients du site ont un profil (`socio_demo`) et une ligne dans `mapping`. `mapping` est bien plus large que le site : 7,83 M de personnes, dont 17,3 % seulement sont venues sur le site. Les autres n'y figurent sans doute qu'à travers les publicités reçues (vérifié en 1.3.4).
-   Publicités : 13 % des impressions programmatiques et 5 % des impressions TV ne peuvent être rattachées à personne : 11 % et 3 % n'ont pas d'identifiant, 2 % ont un identifiant absent de `mapping`.

#### 1.3.4 ) Tables d'analyse

Ces tables sont définies une seule fois ici et réutilisées dans les parties 2 et 3.

**Tables des ventes (partie 2) :**

- `orders_c` : lignes `Order` du niveau B, y compris celles sans client identifié.
- `orders_id` : lignes `Order` du niveau C.
- `orders_id_unique` : une ligne par commande. Une commande regroupe les lignes `Order` d'un même client au même horodatage (à la seconde).
- `customer_order` : une ligne par acheteur : CA, quantité, nombre de lignes et de produits distincts, nombre de commandes, dates de première et de dernière commande.
- `customer` : une ligne par client identifié du site, acheteur ou non, avec son profil socio-démographique. Les indicateurs d'achat valent 0 pour les non-acheteurs.

In [ ]:
orders_c = retailer_c.loc[retailer_c["event_name"] == "Order"].copy() 

orders_id = retailer_id.loc[retailer_id["event_name"] == "Order"].copy()  

# Une ligne par commande
orders_id_unique = (orders_id.groupby(["customer_id", "timestamp_utc"])
                .agg(CA=("sales", "sum"),
                     quantite=("quantity", "sum"),
                     nb_lignes=("sales", "size"))
                .reset_index()) 

# Une ligne par acheteur
customer_order = (orders_id.groupby("customer_id")
          .agg(CA=("sales", "sum"), 
               quantite=("quantity", "sum"),
               nb_lignes=("sales", "size"), 
               nb_produits=("product_name", "nunique"))
          .join(orders_id_unique.groupby("customer_id")
                .agg(nb_commandes=("CA", "size"),
                     premiere_commande=("timestamp_utc", "min"),
                     derniere_commande=("timestamp_utc", "max")))
          .reset_index())

# Une ligne par client du site, acheteur ou non          
customer = (retailer_id[["customer_id"]].drop_duplicates()
           .merge(socio_id, on="customer_id", how="left", validate="one_to_one")
           .merge(customer_order, on="customer_id", how="left", validate="one_to_one"))
cols_achat = ["CA", "quantite", "nb_lignes", "nb_produits", "nb_commandes"]
customer[cols_achat] = customer[cols_achat].fillna(0)
customer["acheteur"] = customer["nb_commandes"] > 0
customer["panier_moyen"] = customer["CA"] / customer["nb_commandes"].replace(0, np.nan)
assert customer["customer_id"].is_unique and customer["age"].notna().all()


**Tables de la mesure publicitaire (partie 3) :**

- `parcours` : la frise de chaque personne, avec une ligne par événement (vue, panier, commande, publicité programmatique par campagne, publicité TV), dans l'ordre chronologique. Elle sert à savoir si une publicité précède une visite ou un achat ;
- `pop` : une ligne par personne de `mapping`, venue sur le site ou non, avec son exposition par canal, son activité sur le site, ses commandes avant et après le début de la TV et son profil. C'est la table qui sert à mesurer l'effet de la publicité.

**`parcours` : règles de construction**

- une ligne par commande (et non par ligne de commande), avec le CA total de la commande ;
- les publicités sont rattachées à une personne via `mapping` ; celles qui ne correspondent à aucun client sont retirées ;
- quand une action sur le site et une publicité ont lieu à la même seconde, l'action est placée avant : une publicité doit précéder strictement une action pour avoir pu la provoquer ;
- les horodatages restent en UTC : l'ordre des événements ne dépend pas du fuseau horaire ;
- l'absence de pub dans la frise ne veut pas dire « non exposé » pour un client non mesurable : la règle NaN s'applique dans `pop`.

In [ ]:
ordre_types = ["Product Page View", "Add to cart", "Order", "Pub Contextual", "Pub Retargeting", "Pub TV"]

# Site : vues et paniers (une ligne par événement)
vues_paniers = retailer_id.loc[retailer_id["event_name"] != "Order"]
site = pd.DataFrame({
    "customer_id": vues_paniers["customer_id"],
    "timestamp_utc": vues_paniers["timestamp_utc"],
    "type": vues_paniers["event_name"],
})

# Site : commandes (une ligne par commande, CA total)
commandes = pd.DataFrame({
    "customer_id": orders_id_unique["customer_id"],
    "timestamp_utc": orders_id_unique["timestamp_utc"],
    "type": "Order",
    "CA": orders_id_unique["CA"],
})

# Publicités : rattachées à une personne via mapping
cle_dsp = mapping_id.dropna(subset=["dsp_id"]).set_index("dsp_id")["customer_id"]
cle_tv = mapping_id.dropna(subset=["device_id"]).set_index("device_id")["customer_id"]

pub_prog = pd.DataFrame({
    "customer_id": prog_id["dsp_id"].map(cle_dsp),
    "timestamp_utc": prog_id["timestamp_utc"],
    "type": "Pub " + prog_id["campaign_name"].astype(str),
}).dropna(subset=["customer_id"])
pub_tv = pd.DataFrame({
    "customer_id": tv_id["device_id"].map(cle_tv),
    "timestamp_utc": tv_id["timestamp_utc"],
    "type": "Pub TV",
}).dropna(subset=["customer_id"])

# Assemblage et tri chronologique (à la même seconde, l'action sur le site passe avant la pub)
parcours = pd.concat([site, commandes, pub_prog, pub_tv], ignore_index=True)
parcours["type"] = pd.Categorical(parcours["type"], categories=ordre_types, ordered=True)
parcours["est_pub"] = parcours["type"].isin(ordre_types[3:])
parcours = parcours.sort_values(["customer_id", "timestamp_utc", "type"], ignore_index=True)
del vues_paniers, site, commandes, pub_prog, pub_tv

**`pop` : une ligne par personne**

Une ligne pour chacune des 7,8 M personnes de `mapping`, qu'elles soient venues sur le site ou non. Colonnes :

- exposition, pour chaque canal (`contextual`, `retargeting`, `tv`) : nombre de publicités (`nb_*`), date de la première (`premiere_*`) et indicateur `expose_*`. Ces colonnes valent NaN si la personne n'est pas mesurable sur ce canal, et 0 (ou False) si elle est mesurable mais non exposée ;
- `profil_expo` : la combinaison de publicités reçues (« Aucune pub », « Contextual + TV »...), pour les personnes mesurables sur les deux canaux ;
- site : nombre de vues, de paniers et de commandes, CA, date de la première visite, indicateurs `visiteur` et `acheteur` ;
- avant / après le début de la TV : commandes et CA de chaque période. La coupure est la première impression TV (`debut_tv`) ;
- `pub_avant_visite_*` : la première publicité du canal précède-t-elle la première visite observée ? NaN si la personne n'est pas visiteuse ou n'a pas été exposée à ce canal ;
- profil (âge, revenu, race de l'animal) : connu pour tous les visiteurs.

In [ ]:
suffixes = {"Product Page View": "vue", "Add to cart": "panier", "Order": "commande",
            "Pub Contextual": "contextual", "Pub Retargeting": "retargeting", "Pub TV": "tv"}
canaux = {"contextual": "dsp_mesurable", "retargeting": "dsp_mesurable", "tv": "tv_mesurable"}

# Nombre d'événements et date du premier, par personne et par type
evenements = (parcours.groupby(["customer_id", "type"], observed=True)["timestamp_utc"]
                      .agg(nb="size", premiere="min").unstack())
evenements.columns = [f"{stat}_{suffixes[t]}" for stat, t in evenements.columns]

# Commandes et CA avant / après le début de la TV, puis CA total (somme des deux périodes)
cmd = parcours.loc[parcours["type"] == "Order", ["customer_id", "timestamp_utc", "CA"]]
cmd["periode"] = np.where(cmd["timestamp_utc"] < debut_tv, "avant", "apres")
periodes = (cmd.groupby(["customer_id", "periode"])["CA"]
               .agg(nb_commandes="size", CA="sum").unstack(fill_value=0))
periodes.columns = [f"{stat}_{p}" for stat, p in periodes.columns]
periodes["CA"] = periodes["CA_avant"] + periodes["CA_apres"]

# Une ligne par personne de mapping
pop = (mapping_id[["customer_id", "dsp_mesurable", "tv_mesurable"]]
       .merge(evenements, left_on="customer_id", right_index=True, how="left")
       .merge(periodes, left_on="customer_id", right_index=True, how="left")
       .merge(socio_id, on="customer_id", how="left", validate="one_to_one"))
assert len(pop) == len(mapping_id) and pop["customer_id"].is_unique

# Activité sur le site : 0 pour les personnes jamais venues
cols_site = ["nb_vue", "nb_panier", "nb_commande", "nb_commandes_avant", "nb_commandes_apres",
             "CA_avant", "CA_apres", "CA"]
pop[cols_site] = pop[cols_site].fillna(0)
pop["premiere_visite"] = pop[["premiere_vue", "premiere_panier", "premiere_commande"]].min(axis=1)
pop["visiteur"] = pop["premiere_visite"].notna()
pop["acheteur"] = pop["nb_commande"] > 0

# Exposition par canal : NaN si la personne n'est pas mesurable sur ce canal, 0 / False si mesurable non exposée
for c, mesurable in canaux.items():
    pop[f"nb_{c}"] = pop[f"nb_{c}"].fillna(0).where(pop[mesurable])
    pop[f"expose_{c}"] = (pop[f"nb_{c}"] > 0).where(pop[mesurable]).astype("boolean")

# La 1re pub du canal précède-t-elle la 1re visite ? (NaN si non visiteur ou non exposé au canal)
for c in canaux:
    pop[f"pub_avant_visite_{c}"] = ((pop[f"premiere_{c}"] < pop["premiere_visite"])
                                    .where(pop["visiteur"] & pop[f"premiere_{c}"].notna())
                                    .astype("boolean"))

# profil_expo : canaux reçus (« Aucune pub » si aucun), seulement pour les personnes mesurables 
# sur les deux canaux ; pour les autres, on ne sait pas ce qu'elles ont reçu sur le canal non mesuré (NaN)
combinaison = pd.Series("", index=pop.index)
for c, nom in [("contextual", "Contextual"), ("retargeting", "Retargeting"), ("tv", "TV")]:
    combinaison += np.where(pop[f"expose_{c}"].fillna(False), f" + {nom}", "")
pop["profil_expo"] = (combinaison.str.removeprefix(" + ").replace("", "Aucune pub")
                      .where(pop["dsp_mesurable"] & pop["tv_mesurable"]))
del evenements, cmd, periodes, combinaison

**Vérifications :** rien n'est perdu entre les tables sources, `parcours` et `pop`.

In [ ]:
nb_par_type = parcours["type"].value_counts().reindex(ordre_types)

# parcours : tous les événements du site et toutes les publicités rattachables (1.3.3)
assert parcours["customer_id"].notna().all() and parcours["type"].notna().all()
assert nb_par_type["Product Page View"] == (retailer_id["event_name"] == "Product Page View").sum()
assert nb_par_type["Add to cart"] == (retailer_id["event_name"] == "Add to cart").sum()
assert nb_par_type["Order"] == len(orders_id_unique)
assert np.isclose(parcours["CA"].sum(), orders_id["sales"].sum())
assert (nb_par_type[["Pub Contextual", "Pub Retargeting"]].sum()
        == rattachement.loc["Programmatique", "Identifiant présent dans mapping"])
assert nb_par_type["Pub TV"] == rattachement.loc["TV", "Identifiant présent dans mapping"]

# pop : mêmes commandes, même CA, mêmes publicités, mêmes visiteurs et acheteurs
assert pop["nb_commande"].sum() == nb_par_type["Order"]
assert (pop["nb_commandes_avant"] + pop["nb_commandes_apres"] == pop["nb_commande"]).all()
assert np.isclose(pop["CA"].sum(), parcours["CA"].sum())
for c, t in [("contextual", "Pub Contextual"), ("retargeting", "Pub Retargeting"), ("tv", "Pub TV")]:
    assert pop[f"nb_{c}"].sum() == nb_par_type[t]
assert pop["visiteur"].sum() == retailer_id["customer_id"].nunique()   
assert pop["acheteur"].sum() == orders_id["customer_id"].nunique()
print("Vérifications : OK\n")
print(f"pop : {len(pop):,} personnes | visiteurs : {pop['visiteur'].sum():,} ({pop['visiteur'].mean():.1%})"
      f" | acheteurs : {pop['acheteur'].sum():,} ({pop['acheteur'].mean():.1%})")

# Personnes sans publicité mesurée ni visite : pourquoi sont-elles dans mapping ?
sans_rien = pop[~pop["visiteur"] & pop[["nb_contextual", "nb_retargeting", "nb_tv"]].fillna(0).sum(axis=1).eq(0)]
print(f"\nPersonnes sans publicité mesurée ni visite : {len(sans_rien):,}")
print(sans_rien[["dsp_mesurable", "tv_mesurable"]].value_counts())

**Récapitulatif :**

- Vérifications : `parcours` contient tous les événements du site et toutes les publicités rattachables (1.3.3) ; `pop` retrouve les mêmes commandes, le même CA et les mêmes publicités. `pop` compte 7,83 M de personnes, dont 1,35 M de visiteurs (17,3 %) et 500 877 acheteurs (6,4 %).
- Composition de `mapping` : 0,54 M de personnes n'ont ni publicité mesurée ni visite. Toutes sauf 10 sont non mesurables sur au moins un canal : elles ont sans doute été exposées sur ce canal. `mapping` ne contient donc que des personnes exposées à une publicité ou venues sur le site : il n'y a pas de groupe « ni exposé, ni visiteur ».

## 2 ) Comprendre le marché

### 2.1 ) Cadre

In [ ]:
print(retailer_c.describe(include="all").round(2))

# --- Période d'étude ---
debut, fin = retailer_c["timestamp_utc"].min(), retailer_c["timestamp_utc"].max()
print(f"\nPériode : {debut} -> {fin} ({(fin - debut).days + 1} jours)")

# --- Répartition des marques et nombre de produits par marque ---
repartition = (retailer_c["brand"].value_counts(normalize=True)*100).round(0).astype(int)
produits = retailer_c.groupby("brand", observed=True)["product_name"].nunique(dropna=False)
tableau = pd.concat([
    repartition.rename("% des lignes"),
    produits.rename("Produits distincts")
], axis=1)
print(f"\n{tableau}")

# --- Nombre de clients (identifiés uniquement) ---
print(f"\nClients identifiés        : {retailer_id['customer_id'].nunique():,}")
print(f"Lignes sans client connu  : {retailer_c['customer_id'].isna().sum():,}")

# --- Répartition des types d'événements ---
print(f"\n{(retailer_c['event_name'].value_counts(normalize=True)*100).round(0).astype(int)}")

# --- CA et quantité totaux (événements Order uniquement) ---
print(f"\nCA total        : {orders_c['sales'].sum():,.0f}")
print(f"\nQuantité totale : {orders_c['quantity'].sum():,.0f}")

**Récapitulatif :**

- Période étudiée : du 1er janvier 2024 au 30 juin 2024 (182 jours = 6 mois)  
- 3 "marques" étudiées : 
    - Hill's (18 produits) : marque mère spécialisée dans la nutrition des chiens et des chats.
    - Science Diet (298 produits) : gamme d'alimentation quotidienne adaptée aux besoins nutritionnels des animaux en bonne santé.
    - Prescription Diet (189 produits) : gamme d'aliments diététiques répondant à des besoins de santé spécifiques.
- Nombre de clients identifiés : 1 354 524
- 3 types d'événements : Product Page View (vue de la page produit), Add to cart (ajout au panier) et Order (achat)  
- CA total : 62 427 795 \$  
- Quantité totale : 1 552 299 produits achetés

### 2.2 ) Comportement d'achat et dynamique des ventes

#### 2.2.1 ) Structure de la clientèle et fréquence d'achat



In [ ]:
# --- Nombre d'acheteurs---
print(f"Clients ayant commandé    : {orders_id['customer_id'].nunique():,}")

# --- Nombre de commandes---
print(f"Nombre de commandes    : {len(orders_id_unique):,}")

# --- Nombre de commandes par acheteur---
repartition = customer_order["nb_commandes"].clip(upper=5).value_counts().sort_index()
repartition.index = [str(i) if i < 5 else "5+" for i in repartition.index]
repartition.index.name = "Nombre de commandes"
print(f"\n{pd.DataFrame({'Clients': repartition, "Part (%)": (repartition / repartition.sum() * 100).round(0).astype(int)})}")

# --- Taux de réachat : part des acheteurs avec au moins 2 commandes ---
taux_reachat = (customer_order["nb_commandes"] >= 2).mean()
print(f"\nTaux de réachat (>= 2 commandes) : {taux_reachat:.0%}")

**Récapitulatif :**

- 500 877 acheteurs qui ont passé 1 404 543 commandes
- Environ 6 clients sur 10 rachètent -> une majorité de clients fidèles, mais une part importante de clients ne commande qu'une fois (41 %).

#### 2.2.2 ) Composition et valeur des commandes


In [ ]:
# --- Nombre de produits par commande---
repartition = orders_id_unique["nb_lignes"].value_counts().sort_index()
tableau = pd.DataFrame({
    "Nombre de commandes": repartition,
    "Part des commandes (%)": (repartition / repartition.sum() * 100).round(2)
})
tableau.index.name = "Nombre de produits par commande"
print(f"\n{tableau}")

# --- Quantité achetée d'un même produit lors d'une même commande ---
repartition = (orders_c["quantity"].map(lambda x: "1" if x == 1 else "2+").value_counts())
repartition.index.name = "Nombre d'unités d'un même produit"
print(((repartition / repartition.sum()) * 100).round(0).astype(int).to_frame("Part (%)"))

# --- Dépenses et comportement d'achat par client ---
customer_order["panier_moyen"] = customer_order["CA"] / customer_order["nb_commandes"]
print("\nStatistiques par acheteur :")
print(customer_order[["CA", "quantite", "nb_produits","nb_commandes", "panier_moyen"]].describe(percentiles=[.25, .5, .75, .9]).round(2))


**Récapitulatif :**

- Taille du panier : selon la définition retenue, la quasi-totalité des commandes sont composées d'un seul produit (potentiellement en plusieurs quantités).
- Quantités : 94 % des achats portent sur une seule unité ; achats multiples relativement rares (6 %).
- Dépenses cumulées : 122 \$ en moyenne par client, mais une forte dispersion (79 \$ en médiane).
- Panier moyen par client : 45 \$ par commande en moyenne (42 \$ en médiane).

**Point de vigilance :** La distribution du CA est très asymétrique (moyenne 122 \$, médiane 79 \$) : les médianes décrivent mieux le client type. De plus, on observe des valeurs atypiques. Par exemple, un client a passé 2 012 commandes en 182 jours...

In [ ]:
# --- Comptes atypiques : règle métier = plus d'une commande par jour en moyenne ---
nb_jours = (fin - debut).days + 1   # 182 jours
customer_order["atypique"] = customer_order["nb_commandes"] > nb_jours
customer["atypique"] = customer["customer_id"].isin(
    customer_order.loc[customer_order["atypique"], "customer_id"])   # réutilisé en partie 2

aty = customer_order[customer_order["atypique"]]
print(aty)
print(f"\nComptes atypiques : {len(aty)} | "
      f"{aty['nb_commandes'].sum() / customer_order['nb_commandes'].sum():.2%} des commandes | "
      f"{aty['CA'].sum() / customer_order['CA'].sum():.2%} du CA")

# Sensibilité : indicateurs par acheteur avec et sans ces comptes
cols = ["CA", "nb_commandes", "panier_moyen"]
hors = customer_order.loc[~customer_order["atypique"], cols]
print(pd.DataFrame({
    "Moyenne": customer_order[cols].mean(),
    "Moyenne hors atyp.": hors.mean(),
    "Écart-type": customer_order[cols].std(),
    "Écart-type hors atyp.": hors.std(),
    "Médiane": customer_order[cols].median(),
}).round(2))

**Récapitulatif :**

- Valeurs extrêmes : 3 comptes passent plus d’une commande par jour en moyenne (jusqu’à 2 012 commandes en 182 jours), ce qui est peu plausible pour un particulier. Il peut s’agir d’un professionnel (élevage, refuge, clinique, revendeur), d’un identifiant partagé ou d’une erreur de collecte. Ils n’ont pas été investigués plus loin.
- Poids : Ces comptes représentent 0,13 % du CA. Ils ne changent pas les moyennes, mais gonflent fortement la dispersion du nombre de commandes (écart-type de 4,4 avec eux, 2,9 sans eux)

**Décision :** ils sont conservés, puisque les ventes ont bien eu lieu et pèsent peu, et repérés par la variable atypique.

#### 2.2.3 ) Analyse des prix et des volumes

In [ ]:
# --- Prix unitaire ---
assert (orders_c["quantity"] > 0).all()
assert (orders_c["sales"] > 0).all()
orders_c["prix_unitaire"] = orders_c["sales"] / orders_c["quantity"]
print(orders_c[["sales", "quantity", "prix_unitaire"]]
      .describe(percentiles=[.25, .5, .75, .95]).round(2))

# --- Distribution des montants et prix unitaires ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
orders_c["sales"].plot(
    kind="hist", bins=50, ax=axes[0]
)
axes[0].set_title("Montant des lignes de commande")
orders_c["prix_unitaire"].plot(
    kind="hist", bins=50, ax=axes[1]
)
axes[1].set_title("Prix unitaire des produits")
plt.tight_layout()
plt.show()

# --- Prix par produit ---
prix_produit = (
    orders_c.groupby("product_name")
    .agg(
        nb_lignes=("sales", "size"),
        quantite=("quantity", "sum"),
        CA=("sales", "sum"),
        prix_median=("prix_unitaire", "median")
    )
)
print("\nPrincipaux produits par chiffre d'affaires :")
print(prix_produit.sort_values("CA", ascending=False).head(10).round(2))

# --- Prix par marque ---
print("\nPrix unitaires par marque :")
print(
    orders_c.groupby("brand", observed=True)["prix_unitaire"]
    .agg(["count", "median", "mean"])
    .sort_values("median", ascending=False)
    .round(2)
)

**Récapitulatif :**

- Prix unitaires : forte variation selon les produits, avec des prix médians allant d'environ 47 \$ à 86 \$ parmi les principaux contributeurs au CA.
- Positionnement des marques : Prescription Diet présente les prix les plus élevés (56 \$ de médiane), devant Science Diet (42 \$) et Hills (8 \$).
- Contribution au CA : quelques références concentrent des chiffres d'affaires importants, notamment les grands conditionnements Science Diet pour chiens adultes.
- Différenciation tarifaire : les prix varient fortement selon les gammes et les formats de conditionnement.

### 2.3 ) Temporalité des ventes

#### 2.3.1 ) Evolution mensuelle

In [ ]:
orders_c["mois"] = orders_c["timestamp_utc"].dt.to_period("M")

par_mois = orders_c.groupby("mois").agg(
    CA=("sales", "sum"),
    commandes=("sales", "count")
)

par_mois["jours"] = par_mois.index.days_in_month
par_mois["CA_par_jour"] = par_mois["CA"] / par_mois["jours"]
par_mois["lignes_par_jour"] = par_mois["commandes"] / par_mois["jours"]

fig, axes = plt.subplots(2, 1, figsize=(12, 7))

par_mois["CA_par_jour"].plot(ax=axes[0])
axes[0].set_title("Évolution mensuelle du chiffre d'affaires journalier moyen")
axes[0].set_ylabel("CA par jour")
axes[0].grid(alpha=0.3)

par_mois["lignes_par_jour"].plot(ax=axes[1])
axes[1].set_title("Évolution mensuelle du nombre moyen de lignes de commande par jour")
axes[1].set_ylabel("Lignes par jour")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# --- Valeurs exactes pour le récapitulatif ---
par_mois["écart_moyenne_%"] = (par_mois["CA_par_jour"] / par_mois["CA_par_jour"].mean() - 1) * 100
par_mois["CA_par_ligne"] = par_mois["CA"] / par_mois["commandes"]
print(par_mois.round(1))

*Remarque : les valeurs sont des moyennes journalières, pour comparer des mois de longueurs différentes (février 2024 compte 29 jours). Les axes ne partent pas de 0 : ils grossissent visuellement des écarts qui restent compris entre −4,3 % et +7,6 % par rapport à la moyenne.*

**Récapitulatif :**

- Une activité globalement stable : environ 343 000  \$ de CA par jour en moyenne sur le semestre. Janvier, mai et juin restent à moins de 1 % de cette moyenne.
- Un pic en février : 369 000  \$ et 8 420 lignes de commande par jour, soit environ +7 % par rapport à la moyenne. Ce pic n'apparaît qu'en moyenne journalière : en total mensuel, février est au niveau de janvier.
- Un creux en mars–avril : environ 329 000  \$ et 7 550 lignes par jour, soit environ −4 % par rapport à la moyenne. L'activité revient à la normale dès mai.
- Des variations dues au nombre de commandes, pas aux prix : les deux courbes ont la même forme, et le montant moyen par ligne reste presque constant (43,2 à 43,9  \$).

#### 2.3.2 ) Choix du fuseau horaire et répartition horaire

Les horodatages sont en UTC. Pour étudier les heures et les jours d'achat, on les convertit en heure de New York, en faisant l'hypothèse d'une clientèle majoritairement américaine :

- les poids des produits sont exprimés en livres (lb) et en onces (oz) ;
- les revenus sont exprimés en milliers de dollars (K  \$) ;
- les États-Unis sont le principal marché de Hill's d'après mes recherches.

La conversion avec `tz_convert("America/New_York")` tient compte du passage à l'heure d'été le 10 mars 2024 (UTC−5, puis UTC−4).

C'est une approximation : les États-Unis couvrent plusieurs fuseaux horaires (3 h d'écart entre les côtes Est et Ouest, sans compter l'Alaska et Hawaï). On retient celui de New York, le fuseau le plus peuplé ; pour les clients de l'Ouest, les heures affichées sont donc décalées.

Le graphique ci-dessous vérifie que cette hypothèse est cohérente. En UTC, le creux d'activité tombe vers 8–9 h, ce qui serait invraisemblable pour une clientèle européenne. En heure de New York, il tombe vers 3–4 h du matin, et la courbe suit un rythme de journée plausible.

À l'échelle du mois, le changement de fuseau ne déplace que quelques heures en début et en fin de mois : l'analyse mensuelle reste en UTC.

In [ ]:
orders_c["timestamp_ny"] = (
    pd.to_datetime(orders_c["timestamp_utc"], utc=True)
    .dt.tz_convert("America/New_York")
)
orders_c["heure_utc"] = orders_c["timestamp_utc"].dt.hour
orders_c["heure_ny"] = orders_c["timestamp_ny"].dt.hour

par_heure = pd.DataFrame({
    ("CA", "UTC"): orders_c.groupby("heure_utc")["sales"].sum(),
    ("CA", "Heure de New York"): orders_c.groupby("heure_ny")["sales"].sum(),
    ("lignes", "UTC"): orders_c.groupby("heure_utc")["sales"].size(),
    ("lignes", "Heure de New York"): orders_c.groupby("heure_ny")["sales"].size(),
})

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

par_heure["CA"].plot(ax=axes[0], marker="o")
axes[0].set_title("CA par heure : UTC et heure de New York")
axes[0].set_ylabel("CA ()")
axes[0].grid(alpha=0.3)

par_heure["lignes"].plot(ax=axes[1], marker="o", legend=False)
axes[1].set_title("Lignes de commande par heure : UTC et heure de New York")
axes[1].set_ylabel("Lignes de commande")
axes[1].set_xlabel("Heure")
axes[1].set_xticks(range(24))
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Récapitulatif :**

- Une hypothèse cohérente : en UTC, l'activité est minimale vers 8 h et maximale vers 23 h–minuit, ce qui serait invraisemblable pour une clientèle européenne. En heure de New York, le creux tombe à 4 h du matin et l'activité reprend à partir de 6 h, ce qui correspond à une journée type.
- Très peu d'achats la nuit.
- Deux pics dans la journée : en fin de matinée (10–11 h), puis en début de soirée (18–19 h), séparés par un léger creux vers 15 h. 
- Des variations dues au nombre de commandes, pas aux prix : les courbes du CA et des lignes de commande ont la même forme.

#### 2.3.3 ) Répartition par jour de la semaine

In [ ]:
jours_fr = ["Lun", "Mar", "Mer", "Jeu", "Ven", "Sam", "Dim"]

# En heure de New York, la période va du 31/12 19 h au 30/06 20 h : on ne garde que les jours complets
jours_ny = (orders_c
            .groupby(orders_c["timestamp_ny"].dt.normalize())
            .agg(CA=("sales", "sum"), lignes=("sales", "size"))
            .loc["2024-01-01":"2024-06-29"])

par_js = (jours_ny
          .groupby(jours_ny.index.day_of_week)[["CA", "lignes"]]
          .mean()
          .set_axis(jours_fr))
par_js["écart_moyenne_%"] = (par_js["CA"] / par_js["CA"].mean() - 1) * 100
print(par_js.round(1))

par_js[["CA", "lignes"]].plot(kind="bar", subplots=True, layout=(2, 1),
                              figsize=(10, 6), legend=False, sharex=True,
                              title=["CA moyen par jour", "Lignes de commande moyennes par jour"])
plt.tight_layout()
plt.show()

**Récapitulatif :**

- Peu d'écart entre les jours : le CA moyen va de 328 000  \$ (samedi) à 356 000  \$ (lundi), soit de −4,3 % à +3,8 % autour de la moyenne (environ 343 000  \$ par jour).
- Un début de semaine plus fort : le lundi est le jour le plus actif, avec 356 000   \$ et 8 090 lignes de commande par jour (+3,8 %).
- Un creux en fin de semaine : le vendredi (−2,2 %) et surtout le samedi, jour le plus faible, avec 328 000   \$ et 7 590 lignes par jour (−4,3 %). Le dimanche repasse au-dessus de la moyenne (+1,9 %).
- Des variations dues au nombre de commandes, pas aux prix : le montant moyen par ligne reste presque constant (43,3 à 44,1   \$).

### 2.4 ) Parcours d'achat sur le site

#### 2.4.1 ) Répartition des clients par étape

In [ ]:
etapes = ["Product Page View", "Add to cart", "Order"]

funnel = retailer_id.groupby("event_name", observed=True)["customer_id"] \
    .nunique().reindex(etapes)

print("Nombre de clients uniques par étape :")
print(funnel)

**Récapitulatif :**

- Volume : forte diminution du nombre de clients entre la consultation d'un produit (1,31 M), l'ajout au panier (620 k) et la commande (501 k).

#### 2.4.2 ) Taux de passage entre les étapes

In [ ]:
vus = set(retailer_id.loc[
    retailer_id["event_name"] == "Product Page View", "customer_id"
])

paniers = set(retailer_id.loc[
    retailer_id["event_name"] == "Add to cart", "customer_id"
])

acheteurs = set(retailer_id.loc[
    retailer_id["event_name"] == "Order", "customer_id"
])

print(f"Vue → panier       : {len(vus & paniers) / len(vus):.0%}")
print(f"Panier → commande  : {len(paniers & acheteurs) / len(paniers):.0%}")
print(f"Vue → commande     : {len(vus & acheteurs) / len(vus):.0%}")

**Récapitulatif :**

- Vue → panier : moins d'un visiteur sur deux ajoute un produit au panier (44 %).
- Panier → commande : la grande majorité des clients ayant ajouté un produit passent commande (80 %).
- Vue → commande : 36 % des clients ayant vu une fiche produit ont commandé au moins une fois.
- Point de rupture : principale déperdition entre la consultation et l'ajout au panier.

*Attention : ce tunnel de conversion est calculé par client sur tout le semestre, pas par session.*

#### 2.4.3 ) Couverture du suivi de navigation

In [ ]:
sans_vue = acheteurs - vus
print(
    f"Acheteurs sans vue enregistrée : "
    f"{len(sans_vue):,} ({len(sans_vue) / len(acheteurs):.1%})"
)

# Évolution mensuelle
premiere_cmd = customer_order.set_index("customer_id")["premiere_commande"]

t = premiere_cmd.dt.to_period("M").to_frame("mois")
t["sans_vue"] = t.index.isin(sans_vue)

print("\nPart des acheteurs sans vue enregistrée par mois (%) :")
print((t.groupby("mois")["sans_vue"].mean() * 100).round(1))

**Récapitulatif :**

- Fiabilité du suivi : seulement 6 % des acheteurs n'ont aucune vue enregistrée ; cette proportion reste relativement stable sur les six mois (5,2 à 8,1 %).

### 2.5 ) Profil des clients et comportement d'achat selon les caractéristiques socio-démographiques

In [ ]:
taux_global = customer["acheteur"].mean() * 100

def profil(var):
    g = customer.groupby(var, observed=True, dropna=False)

    t = g.agg(
        Clients=("acheteur", "size"),
        Acheteurs=("acheteur", "sum"),
        CA=("CA", "sum"),
        Commandes=("nb_commandes", "sum")
    )

    t["Part clients (%)"] = t["Clients"] / t["Clients"].sum() * 100
    t["Taux d'achat (%)"] = t["Acheteurs"] / t["Clients"] * 100
    t["Part CA (%)"] = t["CA"] / t["CA"].sum() * 100
    t["CA par acheteur"] = t["CA"] / t["Acheteurs"]
    t["Commandes par acheteur"] = t["Commandes"] / t["Acheteurs"]
    t["Panier moyen"] = t["CA"] / t["Commandes"]

    return t

def graphe_taux(t, titre):
    ax = t["Taux d'achat (%)"].plot(
        kind="bar", figsize=(7, 4), color="steelblue", rot=0
    )
    ax.axhline(taux_global, color="red", ls="--",
               label=f"Moyenne : {taux_global:.1f} %")
    ax.set_title(titre)
    ax.set_ylabel("Taux d'achat (%)")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

for var, titre in [
    ("breed", "Taux d'achat selon la race"),
    ("age", "Taux d'achat selon l'âge"),
    ("income", "Taux d'achat selon le revenu")
]:
    t = profil(var)

    print(f"\n--- {titre} ---")
    display(t[[
        "Clients", "Part clients (%)", "Taux d'achat (%)",
        "Part CA (%)", "CA par acheteur",
        "Commandes par acheteur", "Panier moyen"
    ]].round(1))

    graphe_taux(t, titre)

**Récapitulatif :**

- Race : comportements assez similaires entre animaux de race pure (36,0 % de taux d'achat) et croisés (38,4 %)
- Âge : taux d'achat le plus élevé chez les 35-45 ans (45,5 %) ; les plus jeunes et les plus de 65 ans achètent moins (environ 30% de taux d'achat).
- Revenu : taux d'achat maximal chez les revenus inférieurs à 40 K  \$ (40,0 %), puis fluctue entre 33,9 % et 39,1 % pour les autres catégories. Les revenus les plus élevés présentent le taux le plus faible (33,9 %).
- Dépenses : les clients aux revenus les plus faibles dépensent davantage par acheteur et passent plus de commandes en moyenne.

#### Analyse croisée de l'âge et du revenu

In [ ]:
# --- Croisement âge et revenu ---
effectifs = pd.crosstab(customer["age"], customer["income"])

taux_croise = pd.crosstab(
    customer["age"],
    customer["income"],
    values=customer["acheteur"].astype(int),
    aggfunc="mean"
) * 100

print("Nombre de clients par catégorie :")
display(effectifs)

# --- Heatmap ---
plt.figure(figsize=(9, 5))

sns.heatmap(
    taux_croise,
    annot=True,
    fmt=".1f",
    cmap="Blues",
    vmin=0,
    vmax=100,
    cbar_kws={"label": "Taux d'achat (%)"}
)

plt.title("Taux d'achat selon l'âge et le revenu")
plt.xlabel("Revenu annuel")
plt.ylabel("Âge")
plt.tight_layout()
plt.show()

**Récapitulatif :**

- les écarts liés à l'âge restent visibles quel que soit le niveau de revenu. Les 35-45 ans présentent les taux d'achat les plus élevés dans toutes les catégories de revenus (41,7 à 49,3 %). 
- Le revenu semble avoir un effet moins marqué.

### Synthèse du marché

Les constats portent sur les ventes du site (`retailer`, niveaux B et C) et sur le profil des clients (`socio_demo`), du 1er janvier au 30 juin 2024.

Sur ces six mois, le site réalise 62,4 M \$ de chiffre d'affaires : 1,40 M de commandes, passées par 501 k acheteurs, soit 37 % des 1,35 M de clients identifiés. Le marché est stable et concentré sur Science Diet. Les achats sont simples (un produit, une unité, environ 42  \$) et répétés (59 % des acheteurs commandent au moins deux fois). Les variations, dans le temps comme d'un profil à l'autre, viennent du nombre d'acheteurs et de commandes, pas du montant des commandes.

| Constat | Chiffres clés | Section |
|---|---|---|
| **Offre** : trois marques Hill's, largement dominées par Science Diet ; les références qui pèsent le plus dans le CA sont de grands formats pour chiens adultes | Science Diet : 82 % des lignes (298 produits) ; Prescription Diet : 13 % (189) ; Hill's : 5 % (18) | 2.1, 2.2.3 |
| **Prix** : trois niveaux de prix, selon la gamme et le format | Prix unitaire médian : Prescription Diet 56  \$, Science Diet 42  \$, Hill's 8  \$ | 2.2.3 |
| **Commandes** : des achats simples, d'un seul produit (selon la définition retenue) et d'une seule unité | 94 % des lignes portent sur une unité ; panier médian de 42  \$| 2.2.2 |
| **Fidélité** : une majorité d'acheteurs récurrents ; 3 comptes atypiques, conservés et repérés | 59 % des acheteurs commandent au moins deux fois et passent 85 % des commandes ; en médiane, 2 commandes et 79  \$ par acheteur ; comptes atypiques : 0,13 % du CA | 2.2.1, 2.2.2 |
| **Saisonnalité** : pas de saisonnalité marquée ; un pic en février, un creux en mars–avril, un retour à la normale en mai | Environ 343 000  \$ par jour ; de −4,3 % (avril) à +7,6 % (février) autour de la moyenne ; CA par ligne stable (43,2 à 43,9  \$) | 2.3.1 |
| **Rythme d'achat** (heure de New York) : des achats en journée, peu d'écart entre les jours de la semaine | Creux à 4 h, pics à 10–11 h et à 18–19 h ; de −4,3 % (samedi) à +3,8 % (lundi) | 2.3.2, 2.3.3 |
| **Parcours** : la principale perte se situe entre la vue produit et l'ajout au panier | Vue → panier : 44 % ; panier → commande : 80 % ; 6 % d'acheteurs sans vue enregistrée | 2.4 |
| **Profil** : l'âge est le facteur le plus discriminant ; le revenu et la race de l'animal jouent peu ; le panier moyen ne dépend pas du profil | Taux d'achat moyen de 37 % ; 45,5 % chez les 35–45 ans contre environ 30 % chez les 18–25 ans et les 65 ans et plus, à tous les niveaux de revenu ; panier moyen de 42,9 à 44,4  \$ selon le segment | 2.5 |


Trois points à retenir pour mesurer l'effet de la publicité :

-   Le calendrier : la TV démarre le 29 février, juste après le pic de février et avant le creux de mars–avril. Le CA par jour baisse d'environ 6 % après cette date. Comparer simplement les ventes avant et après mélangerait donc l'effet de la TV et ce creux : il faudra comparer l'évolution des personnes exposées à celle des non-exposées.
-   Les achats répétés : 85 % des commandes viennent de clients qui achètent plusieurs fois. Un client exposé qui achète aurait donc pu acheter sans publicité. C'est surtout vrai pour le retargeting, qui vise des personnes déjà venues sur le site.
-   L'âge : les 35-45 ans achètent plus souvent que les 18-25 ans et les plus de 65 ans (45,5 % contre environ 30 %). Il faudra en tenir compte quand on compare des groupes.

## 3 ) Comprendre l’impact de la publicité sur les ventes

**Question :** les personnes exposées à la publicité achètent-elles plus qu'elles ne l'auraient fait sans publicité ?


Ce qu'une personne exposée aurait acheté sans publicité, on ne peut pas l'observer. On compare donc des personnes exposées et non exposées, en tenant compte des trois points vus en fin de partie 2 : le ciblage (3.2 et 3.3.2), le calendrier (3.4) et l'âge (3.3.1).


**Démarche :**

| Étape | Question | Section |
|---|---|---|
| Les campagnes | Combien a-t-on dépensé, quand, et pour toucher qui ? | 3.1 |
| Les groupes comparables | Qui est exposé, et quels groupes peut-on comparer ? | 3.2 |
| La comparaison entre visiteurs | Les visiteurs exposés achètent-ils plus que les autres ? | 3.3 |
| Le lancement de la TV | Les ventes des exposés à la TV augmentent-elles après son lancement ? | 3.4 |

Les analyses par personne utilisent les tables `pop` et `parcours`.

### 3.1 ) Les campagnes publicitaires

Deux tables décrivent la publicité diffusée pendant la période :

| Table | Canal | Identifiant | Variables propres |
|---|---|---|---|
| `tv_pub` | Publicité TV, vue sur une box TV | `device_id` : une box, donc un foyer, et non une personne | — |
| `programmatic_pub` | Publicité en ligne achetée en programmatique | `dsp_id` : un internaute | `campaign_name`, `device_type` |


**Lecture des tables :**

- une ligne correspond à une impression, c'est-à-dire une publicité vue une fois : `cost_milli_cent` est donc le coût de cette impression ;
- 1 milli-cent = 1/1 000 de cent = 1/100 000  \$. Les coûts sont aussi exprimés en CPM, le coût pour 1 000 impressions : CPM = `cost_milli_cent` / 100, en dollars ;
- `campaign_name` prend deux valeurs : *Contextual*, la publicité s'affiche selon le contenu de la page visitée, quel que soit l'internaute ; *Retargeting*, elle est montrée à des internautes déjà passés sur le site.

**Fuseau horaire :** Comme pour les ventes, les analyses par jour et par heure sont en heure de New York (2.3.2).

#### 3.1.1 ) Vue d'ensemble

In [ ]:
# Impressions et budget par campagne (niveau B : toutes les impressions, avec ou sans identifiant)
campagnes = prog_c.groupby("campaign_name", observed=True)["cost"].agg(impressions="size", budget="sum")
campagnes.index = campagnes.index.astype(str)
campagnes.loc["TV"] = [len(tv_c), tv_c["cost"].sum()]
campagnes.loc["Total"] = campagnes.sum()

budget_media = pd.DataFrame({
    "Impressions (M)": campagnes["impressions"] / 1e6,
    "Budget (k$)": campagnes["budget"] / 1e3,
    "Part du budget (%)": campagnes["budget"] / campagnes.loc["Total", "budget"] * 100,
    "CPM ($)": campagnes["budget"] / campagnes["impressions"] * 1000,
})
display(budget_media.style.format(precision=1, decimal=","))
print(f"Budget média / CA du site : {campagnes.loc['Total', 'budget'] / orders_c['sales'].sum():.2%}\n")

# TV : prix de l'impression, en milli-cents
print(tv_c["cost_milli_cent"].value_counts())

**Récapitulatif :**

- Budget : 228 400 \$ sur le semestre pour 23,3 M d'impressions, soit 0,37 % du CA du site.
- TV : 59 % du budget pour un quart des impressions. Le prix est fixe (2 325,51 milli-cents par impression, à 14 impressions près), soit un CPM de 23,3 \$, environ 6 fois celui du contextuel.
- Programmatique : le contextuel représente 78 % des impressions programmatiques (13,6 M) pour 51 K\$, avec un CPM de 3,7 \$. Le retargeting coûte près de 3 fois plus cher à l'impression (CPM d'environ 11 \$) : 3,9 M d'impressions pour 42 K\$.
- Budget non mesurable : 13 % des impressions programmatiques et 5 % des impressions TV ne sont rattachées à aucune personne (1.3.3). Elles ont été payées, mais ne pourront pas être reliées à un achat.

#### 3.1.2 ) Calendrier de diffusion

In [ ]:
# Heure de New York, comme pour les ventes (2.3.2)
for df in (prog_c, tv_c):
    df["timestamp_ny"] = df["timestamp_utc"].dt.tz_localize("UTC").dt.tz_convert("America/New_York")

# Impressions et budget par jour, sur les jours complets (1er janvier - 29 juin)
prog_jour = (prog_c.groupby([pd.Grouper(key="timestamp_ny", freq="D"), "campaign_name"], observed=True)
             .agg(impressions=("cost", "size"), budget=("cost", "sum"))
             .unstack().loc["2024-01-01":"2024-06-29"])
tv_jour = (tv_c.groupby(pd.Grouper(key="timestamp_ny", freq="D"))
           .agg(impressions=("cost", "size"), budget=("cost", "sum"))
           .loc["2024-01-01":"2024-06-29"])

debut_tv_ny = debut_tv.tz_localize("UTC").tz_convert("America/New_York")   # debut_tv : défini en 1.3.3
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
for ax, mesure, titre in zip(axes, ["impressions", "budget"], ["Impressions par jour", "Budget par jour ($)"]):
    for camp in ["Contextual", "Retargeting"]:
        ax.plot(prog_jour.index, prog_jour[(mesure, camp)], label=f"Programmatique - {camp}")
    ax.plot(tv_jour.index, tv_jour[mesure], label="TV")
    ax.axvline(debut_tv_ny, color="grey", ls="--", label="Début de la TV")
    ax.set_title(titre)
    ax.grid(alpha=.3)
axes[0].legend()
plt.tight_layout()
plt.show()

*Remarque : comme pour les ventes, les jours sont en heure de New York (2.3.2).*

**Récapitulatif :**

- Contextuel : une très forte diffusion les dix premiers jours de janvier (environ 230 000 impressions par jour), puis de 50 000 à 100 000 par jour.
- Retargeting : une nette baisse de mi-mars à début avril (environ 50 \$ par jour, contre 200 à 400 \$ le reste du temps).
- TV : une diffusion par vagues, la plus forte fin mars (jusqu'à environ 4 000 \$ par jour), une autre fin mai, et une interruption d'environ deux semaines en juin. Dès son lancement, la TV domine les dépenses quotidiennes.
- Budget et impressions : pour la TV, les deux courbes ont la même forme, car le prix de l'impression est fixe (3.1.1). En programmatique, le prix varie dans le temps et les deux courbes peuvent s'écarter : début janvier, le contextuel obtient environ 3 fois plus d'impressions pour à peu près le même budget.
- Canaux liés dans le temps : pendant la vague TV de fin mars, le contextuel et le retargeting sont au plus bas. À ce moment-là, les effets des différents canaux seront difficiles à séparer.

#### 3.1.3 ) Portée et fréquence d'exposition

In [ ]:
# Personnes exposées par canal (pop : NaN si la personne n'est pas mesurable sur ce canal)
portee = pd.DataFrame({
    "Personnes mesurables": [pop[f"expose_{c}"].notna().sum() for c in canaux],
    "Personnes exposées": [pop[f"expose_{c}"].sum() for c in canaux],
}, index=["Contextual", "Retargeting", "TV"])
display(portee.style.format(thousands=" "))

# Nombre d'impressions par identifiant exposé
n_prog = prog_id.groupby(["dsp_id", "campaign_name"], observed=True).size().unstack(fill_value=0)
freq = pd.DataFrame({"Programmatique": n_prog.sum(axis=1).describe(percentiles=[.5, .9, .99]),
                     "TV": tv_id.groupby("device_id").size().describe(percentiles=[.5, .9, .99])})
display(freq.round(1))

# Qui reçoit quelle(s) campagne(s) ? (part des internautes exposés, en %)
print((n_prog.gt(0).value_counts(normalize=True) * 100).round(1))


**Récapitulatif :**

- Portée : 3,41 M de personnes ont vu au moins une publicité contextuelle, 1,78 M une publicité de retargeting et 2,02 M une publicité TV.
- Une exposition souvent unique : la moitié des identifiants exposés n'a vu qu'une publicité ; 90 % en ont vu au plus 7 (programmatique) ou 6 (TV). Les maximums (194 pour un internaute, 234 pour une box) restent plausibles : 1 à 2 publicités par jour.
- Des publics presque séparés : 64 % des internautes exposés n'ont vu que du contextuel, 32 % que du retargeting et 4 % les deux.

### 3.2 ) Qui est exposé ? Quels groupes comparer ?

Chaque personne mesurable sur les deux canaux est classée selon la combinaison de publicités reçues sur la période (`profil_expo`, 1.3.4). Le groupe « Aucune pub » sert de référence. On se limite aux personnes mesurables sur les deux canaux : pour classer quelqu'un dans « Aucune pub » ou « TV », il faut savoir qu'il n'a pas vu de publicité programmatique, et inversement. Les visiteurs exclus (13,5 %) achètent comme les autres (36,8 % d'acheteurs contre 37,0 %) : la restriction ne devrait pas biaiser la comparaison. On regarde aussi qui reçoit le retargeting, censé viser des personnes déjà passées sur le site.

In [ ]:
# Personnes mesurables sur les deux canaux, selon la combinaison de publicités reçues
m = pop[pop["dsp_mesurable"] & pop["tv_mesurable"]]

g = m.groupby("profil_expo")
comparaison = pd.DataFrame({
    "Personnes": g.size(),
    "Visiteurs (%)": g["visiteur"].mean() * 100,
    "Acheteurs (%)": g["acheteur"].mean() * 100,
    "CA par personne": g["CA"].mean(),
}).sort_values("Personnes", ascending=False)
display(comparaison.round(2))

# Qui est retargeté ?
retargetes = pop[pop["expose_retargeting"].fillna(False)]
print(f"Personnes retargetées : {len(retargetes):,}")
print(f"  jamais venues sur le site pendant la période : {(~retargetes['visiteur']).mean():.1%}")
print(f"  1re pub de retargeting après la 1re visite    : "
      f"{1 - retargetes['pub_avant_visite_retargeting'].mean():.1%} (parmi les visiteurs)")

# Les visiteurs exclus (non mesurables sur au moins un canal) achètent-ils comme les autres ?
vis = pop[pop["visiteur"]]
deux_canaux = (vis["dsp_mesurable"] & vis["tv_mesurable"]).rename("Mesurable sur les deux canaux")
print(vis.groupby(deux_canaux)["acheteur"].agg(Visiteurs="size", Taux_achat="mean").round(3))

**Récapitulatif :**

- Une référence composée uniquement de visiteurs : les groupes exposés comptent de 0,85 % à 13,4 % d'acheteurs, contre 41,7 % pour « Aucune pub ». Ce n'est pas un effet négatif de la publicité mais un effet de construction de la base : `mapping` ne contient que des personnes exposées à une publicité ou venues sur le site (1.3.4). Les non-exposés jamais venus n'y figurent pas, alors que les exposés jamais venus sont nombreux.
- Un retargeting qui suit l'intérêt : sur 1,78 M de personnes retargetées, 74 % ne sont jamais venues sur le site pendant la période (le ciblage repose sans doute sur des visites antérieures au 1er janvier, ce que les données ne permettent pas de vérifier). Parmi celles qui sont venues, près de 3 sur 4 (73,6 %) ont reçu leur première publicité de retargeting après leur première visite.

**Conséquences pour la suite :**

- les comparaisons se font **entre visiteurs**, seule population où exposés et non-exposés sont tous présents. Elles mesurent l'effet de la publicité sur l'achat des personnes qui viennent sur le site, pas sur le fait de venir ;
- « première visite » signifie première visite observée, à partir du 1er janvier.
- l'écart entre retargetés et non-retargetés mélangera l'effet de la publicité et un intérêt déjà présent.

### 3.3 ) Les visiteurs exposés achètent-ils plus ?

#### 3.3.1 ) Comparaison selon la publicité reçue

On compare les visiteurs mesurables sur les deux canaux selon la combinaison de publicités reçues. 

Pour vérifier que les écarts ne viennent pas du profil, on calcule aussi des ratios standardisés sur l'âge et le revenu (standardisation indirecte sur les 30 cases âge × revenu) : nombre d'acheteurs observés / nombre d'acheteurs attendus si le groupe achetait comme les visiteurs sans pub de même âge et de même revenu. Même calcul pour le CA. Un ratio supérieur à 1 signifie que le groupe achète plus que la référence.

In [ ]:
visiteurs = m[m["visiteur"]].copy()
assert visiteurs[["age", "income"]].notna().all().all()   # profil connu pour tous les visiteurs
ref = visiteurs[visiteurs["profil_expo"] == "Aucune pub"]

# Taux d'achat et CA moyen de la référence (visiteurs sans pub), par case âge x revenu
taux_ref = (ref.groupby(["age", "income"], observed=True)
               .agg(p_achat_ref=("acheteur", "mean"), ca_moyen_ref=("CA", "mean")).reset_index())

# Acheteurs et CA attendus dans chaque groupe, case par case
cases = (visiteurs.groupby(["profil_expo", "age", "income"], observed=True)
                  .agg(n=("acheteur", "size"), acheteurs=("acheteur", "sum"), CA=("CA", "sum"), commandes=("nb_commande", "sum"))
                  .reset_index()
                  .merge(taux_ref, on=["age", "income"], how="left"))
assert cases["p_achat_ref"].notna().all()
cases["acheteurs_attendus"] = cases["n"] * cases["p_achat_ref"]
cases["CA_attendu"] = cases["n"] * cases["ca_moyen_ref"]

# Ratios bruts et standardisés (référence = 1)
t = cases.groupby("profil_expo")[["n", "acheteurs", "acheteurs_attendus", "CA", "CA_attendu", "commandes"]].sum()
comparaison_visiteurs = pd.DataFrame({
    "Personnes": t["n"],
    "Part (%)": t["n"] / t["n"].sum() * 100,
    "Acheteurs (%)": t["acheteurs"] / t["n"] * 100,
    "CA par personne": t["CA"] / t["n"],
    "CA par acheteur": t["CA"] / t["acheteurs"],
    "Commandes par acheteur": t["commandes"] / t["acheteurs"],
    "Ratio acheteurs (x)": t["acheteurs"] / t["n"] / ref["acheteur"].mean(),
    "Ratio acheteurs standardisé (x)": t["acheteurs"] / t["acheteurs_attendus"],
    "Ratio CA (x)": t["CA"] / t["n"] / ref["CA"].mean(),
    "Ratio CA standardisé (x)": t["CA"] / t["CA_attendu"],
}).sort_values("Personnes", ascending=False)
print(f"Visiteurs mesurables sur les deux canaux : {len(visiteurs):,}")
comparaison_visiteurs.round(2)

**Récapitulatif :**

- Référence : 35 % des visiteurs n'ont reçu aucune publicité mesurée ; 41,7 % d'entre eux achètent, pour 46,75 \$ de CA par personne.
- Retargeting : les visiteurs retargetés achètent plus (48 % d'acheteurs, 66 \$ de CA par personne), soit 1,15 fois plus d'acheteurs et 1,4 fois plus de CA que la référence.
- Contextuel : les visiteurs exposés achètent près de 4 fois moins (11 % d'acheteurs, ratio de 0,27). Ajouter le contextuel fait baisser la part d'acheteurs dans toutes les combinaisons.
- TV : un écart faible (45 % d'acheteurs contre 42 %, 47 \$ de CA par personne dans les deux cas).
- Profil : les ratios standardisés sont égaux aux ratios bruts à 0,01 près. L'âge et le revenu n'expliquent aucun de ces écarts : l'exposition ne semble pas dépendre du profil socio-démographique.
-   Montant des commandes : une commande rapporte à peu près la même somme dans tous les groupes (entre 42 et 45  \$). Les écarts de CA viennent donc du nombre d'acheteurs et du nombre de commandes : un acheteur retargeté passe 3,1 commandes en moyenne, contre 2,6 dans la référence. Cela va dans le sens d'un ciblage de clients déjà fidèles.

**Point de vigilance :** ces écarts ne sont pas encore des effets de la publicité. Ils peuvent venir du ciblage ou de l'ordre dans le temps, puisqu'un achat peut précéder la publicité.

#### 3.3.2 ) La publicité arrive-t-elle avant l'achat ?

Pour chaque canal, on regarde si la publicité arrive avant la première visite et avant le premier achat. 
On compare ensuite l'achat des visiteurs exposés à un seul canal (sinon on ne saurait pas quel canal compte), selon que la publicité a précédé ou suivi leur première visite. Les visiteurs exposés à un seul canal représentent 78 % des exposés au contextuel, 82 % des retargetés et 65 % des exposés à la TV.

Un visiteur arrivé tard dans la période a moins de temps pour acheter : les ratios sont donc standardisés sur le mois de la première visite (référence : visiteurs sans pub).

In [ ]:
canaux_noms = {"contextual": "Contextual", "retargeting": "Retargeting", "tv": "TV"}

# 1. La pub arrive-t-elle avant la 1re visite et avant le 1er achat ? (visiteurs exposés au canal)
ordre = {}
for c, nom in canaux_noms.items():
    expo = visiteurs[visiteurs[f"expose_{c}"] == True]
    ach = expo[expo["acheteur"]]
    ordre[nom] = {
        "Visiteurs exposés": len(expo),
        "Pub avant la 1re visite (%)": expo[f"pub_avant_visite_{c}"].mean() * 100,
        "Acheteurs exposés": len(ach),
        "Pub avant le 1er achat (%)": (ach[f"premiere_{c}"] < ach["premiere_commande"]).mean() * 100,
    }
display(pd.DataFrame(ordre).round(1))

# 2. Achat des visiteurs exposés à un seul canal, selon que la pub précède ou suit la 1re visite
visiteurs["mois_1re_visite"] = visiteurs["premiere_visite"].dt.to_period("M")
ref = visiteurs[visiteurs["profil_expo"] == "Aucune pub"]
ref_mois = ref.groupby("mois_1re_visite").agg(p_achat_ref=("acheteur", "mean"), ca_moyen_ref=("CA", "mean"))

def ligne(df):
    "Taux d'achat et CA d'un groupe, bruts et standardisés sur le mois de la 1re visite."
    par_mois = df.groupby("mois_1re_visite").agg(n=("acheteur", "size")).join(ref_mois, how="left")
    assert par_mois["p_achat_ref"].notna().all()
    return pd.Series({
        "Personnes": len(df),
        "Acheteurs (%)": df["acheteur"].mean() * 100,
        "Ratio acheteurs (x)": df["acheteur"].mean() / ref["acheteur"].mean(),
        "Ratio acheteurs standardisé (x)": df["acheteur"].sum() / (par_mois["n"] * par_mois["p_achat_ref"]).sum(),
        "CA par personne": df["CA"].mean(),
        "Ratio CA (x)": df["CA"].mean() / ref["CA"].mean(),
        "Ratio CA standardisé (x)": df["CA"].sum() / (par_mois["n"] * par_mois["ca_moyen_ref"]).sum(),
    })

groupes = {"Aucune pub": ref}
for c, nom in canaux_noms.items():
    seul = visiteurs[visiteurs["profil_expo"] == nom]
    groupes[f"{nom} seul, pub avant la 1re visite"] = seul[seul[f"pub_avant_visite_{c}"] == True]
    groupes[f"{nom} seul, pub après la 1re visite"] = seul[seul[f"pub_avant_visite_{c}"] == False]

pd.DataFrame({nom: ligne(df) for nom, df in groupes.items()}).T.round(2)


**Récapitulatif :**

-   Ordre dans le temps : la publicité précède le premier achat pour 58 % des acheteurs exposés au contextuel, 42 % des retargetés et 26 % des exposés à la TV (lancée le 29 février). Pour les autres, le premier achat a eu lieu avant toute publicité du canal.
-   Contextuel : les visiteurs exposés achètent près de 4 fois moins que la référence, que la publicité ait précédé leur première visite (ratio standardisé de 0,26) ou l'ait suivie (0,28). Ce faible taux d'achat caractérise le public touché par le contextuel ; rien n'indique que la publicité augmente les achats.
-   Retargeting : les visiteurs retargetés avant leur première visite achètent comme la référence (0,98). L'écart positif vient de ceux retargetés après (1,10 en acheteurs, 1,25 en CA), donc de visiteurs déjà venus : on ne peut pas séparer un éventuel effet de rappel de l'intérêt déjà présent.
-   TV : une fois standardisés, les deux groupes achètent environ 10 % de plus que la référence (1,13 et 1,09) et génèrent 6 à 7 % de CA en plus. Sans standardisation, les exposés avant leur première visite semblaient acheter moins (0,84) : arrivés après le 29 février, ils ont eu moins de temps pour acheter.

### 3.4 ) Effet de la TV : avant / après le 29 février

La TV est le seul canal lancé en cours de période : avant le 29 février, personne n'y est exposé. On peut donc comparer l'évolution des ventes avant et après son lancement. 
Une comparaison brute ne suffit pas, car les ventes varient aussi sans publicité (2.3.1) : on compare l'évolution des exposés à celle des non-exposés (différence de différences). Cette méthode suppose que les deux groupes auraient évolué de la même façon sans la TV ; on le vérifie avant le 29 février.

La coupure est la première impression TV (`debut_tv`, 1.3.3). Les montants sont rapportés au nombre de jours de chaque période (59,9 jours avant, 122,1 jours après).

#### 3.4.1 ) Ventes des visiteurs exposés et non exposés à la TV

On compare d'abord tous les visiteurs, puis les seuls visiteurs déjà venus avant le 29 février, pour écarter l'effet des nouveaux visiteurs. On vérifie enfin que la pression programmatique a évolué de la même façon dans les deux groupes.

In [ ]:
# Au niveau du site : CA par jour avant / après le début de la TV (niveau B, toutes les commandes)
apres_tv = orders_c["timestamp_utc"] >= debut_tv
ca_jour_site = pd.Series({"avant": orders_c.loc[~apres_tv, "sales"].sum() / jours_avant,
                          "apres": orders_c.loc[apres_tv, "sales"].sum() / jours_apres})
print(f"CA du site par jour : {ca_jour_site['avant']:,.0f} $ avant, {ca_jour_site['apres']:,.0f} $ après "
      f"({ca_jour_site['apres'] / ca_jour_site['avant'] - 1:+.1%})")

def did_tv(df, titre):
    "Différence de différences sur le CA par personne et par jour, exposés TV / non exposés."
    d = df.assign(
        groupe=np.where(df["expose_tv"] == True, "Exposés TV", "Non exposés"),
        acheteur_avant=df["nb_commandes_avant"] > 0,
        acheteur_apres=df["nb_commandes_apres"] > 0,
        ca_jour_avant=df["CA_avant"] / jours_avant,
        ca_jour_apres=df["CA_apres"] / jours_apres,
    )
    d["evolution"] = d["ca_jour_apres"] - d["ca_jour_avant"]
    g = d.groupby("groupe")
    t = pd.DataFrame({
        "Personnes": g.size(),
        "Acheteurs avant (%)": g["acheteur_avant"].mean() * 100,
        "Acheteurs après (%)": g["acheteur_apres"].mean() * 100,
        "CA / pers. / jour avant": g["ca_jour_avant"].mean(),
        "CA / pers. / jour après": g["ca_jour_apres"].mean(),
        "Évolution": g["evolution"].mean(),
    })
    e = d.loc[d["groupe"] == "Exposés TV", "evolution"]
    n = d.loc[d["groupe"] == "Non exposés", "evolution"]
    did = e.mean() - n.mean()
    se = np.sqrt(e.var() / len(e) + n.var() / len(n))
    sans_tv = t.loc["Exposés TV", "CA / pers. / jour avant"] + t.loc["Non exposés", "Évolution"]
    print(f"\n{titre}")
    display(t.round(3))
    print(f"Effet TV : {did:+.3f} $ par personne et par jour "
          f"[IC 95 % : {did - 1.96 * se:+.3f} ; {did + 1.96 * se:+.3f}], "
          f"soit {did / sans_tv:+.1%} du CA attendu sans TV")
    print(f"CA supplémentaire après le début de la TV : {did * jours_apres:+.2f} $ par exposé, "
          f"{did * jours_apres * len(e):+,.0f} $ pour les {len(e):,} personnes exposées")
    return did * jours_apres * len(e)

ca_sup = did_tv(visiteurs, "Tous les visiteurs")
did_tv(visiteurs[visiteurs["premiere_visite"] < debut_tv], "Visiteurs déjà venus avant le début de la TV")

budget_tv = tv_c["cost"].sum()
print(f"\nBudget TV : {budget_tv:,.0f} $ | CA supplémentaire / budget : {ca_sup / budget_tv:.2f} $ par $ dépensé")

# D'où vient l'écart ? Visiteurs venus pour la première fois après le début de la TV
nouveau = visiteurs["premiere_visite"] >= debut_tv
expose = visiteurs["expose_tv"] == True
print(f"\nVisiteurs venus pour la 1re fois après le début de la TV : {nouveau[expose].mean():.0%} des exposés, "
      f"{nouveau[~expose].mean():.0%} des non-exposés")
print(f"Parmi ces nouveaux visiteurs exposés, 1re pub TV après la 1re visite : "
      f"{(visiteurs.loc[nouveau & expose, 'pub_avant_visite_tv'] == False).mean():.0%}")


In [ ]:
groupe = pd.Series(np.where(visiteurs["expose_tv"] == True, "Exposés TV", "Non exposés"),
                   index=visiteurs["customer_id"])
taille = groupe.value_counts()

# CA par personne et par semaine, dans chaque groupe
cmd_sem = parcours.loc[parcours["type"] == "Order", ["customer_id", "timestamp_utc", "CA"]]
cmd_sem["groupe"] = cmd_sem["customer_id"].map(groupe)
cmd_sem = cmd_sem.dropna(subset=["groupe"])
cmd_sem["semaine"] = cmd_sem["timestamp_utc"].dt.to_period("W").dt.start_time
hebdo = cmd_sem.groupby(["semaine", "groupe"])["CA"].sum().unstack() / taille

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
hebdo.plot(ax=axes[0], marker="o")
axes[0].set_title("CA par personne et par semaine"); axes[0].set_ylabel("$")
(hebdo["Exposés TV"] - hebdo["Non exposés"]).plot(ax=axes[1], marker="o", color="black")
axes[1].axhline(0, color="grey", lw=0.8)
axes[1].set_title("Écart exposés TV - non exposés"); axes[1].set_ylabel("$")
for ax in axes:
    ax.axvline(debut_tv, color="red", ls="--", label="Début TV")
    ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

# Pression programmatique avant / après, dans chaque groupe (impressions par personne et par jour)
pubs_prog = parcours.loc[parcours["type"].isin(["Pub Contextual", "Pub Retargeting"]), ["customer_id", "timestamp_utc"]]
pubs_prog["groupe"] = pubs_prog["customer_id"].map(groupe)
pubs_prog = pubs_prog.dropna(subset=["groupe"])
pubs_prog["periode"] = np.where(pubs_prog["timestamp_utc"] < debut_tv, "avant", "apres")
pression = pubs_prog.groupby(["groupe", "periode"]).size().unstack().div(taille, axis=0)
pression["avant"] /= jours_avant
pression["apres"] /= jours_apres
pression["Évolution (%)"] = (pression["apres"] / pression["avant"] - 1) * 100
print("Impressions programmatiques par personne et par jour :")
display(pression[["avant", "apres", "Évolution (%)"]].round(4))


**Récapitulatif :**

- Site : le CA journalier passe d'environ 357 000 \$ avant le lancement de la TV à 336 000 \$ après (−6 %). Cette baisse brute ne dit rien de l'effet de la TV : elle suit le pic de février et le creux de mars–avril (2.3.1).
- Tendances parallèles : avant le 29 février, exposés et non-exposés ont le même CA par personne et par jour (environ 0,25 \$) et l'écart hebdomadaire oscille autour de 0. L'hypothèse de la méthode est plausible.
- Tous les visiteurs : après le 29 février, l'écart devient positif et le reste jusqu'en juin. L'effet estimé est de +0,034 \$ par personne et par jour (+14 \% du CA attendu sans TV), soit 417 654 \$ de CA supplémentaire, ou 3 \$ de CA par dollar de budget TV.
- Visiteurs déjà venus avant le 29 février : le résultat s'inverse (−0,017 \$ par personne et par jour, soit −4 %). Les deux groupes baissent d'environ un tiers, ce qui est attendu : ils ont été retenus parce qu'ils étaient actifs avant. Chez les clients déjà présents, la TV n'a pas d'effet visible.
- Origine de l'écart : l'effet positif vient des visiteurs arrivés après le 29 février, plus nombreux chez les exposés (plus de la moitié, contre 46 % chez les non-exposés). La TV a pu les amener, mais pour près de la moitié d'entre eux, la première publicité TV est arrivée après leur première visite. Les 418 000 \$ ne peuvent donc pas être attribués à la TV.
- Programmatique : la pression programmatique baisse environ de moitié dans les deux groupes après le 29 février ; elle n'explique pas l'écart.

#### 3.4.2 ) La TV fait-elle venir de nouveaux clients ?

Les visiteurs ont été retenus parce qu'ils sont venus sur le site : cette population ne permet pas de savoir si la TV fait venir de nouvelles personnes. On reprend donc la comparaison sur les personnes exposées au contextuel, présentes dans `mapping` qu'elles soient venues ou non sur le site (les non-visiteurs comptent pour 0 \$) : leur présence dans la base ne dépend pas d'une visite. On compare le CA par personne, puis le taux de premières visites et de premiers achats parmi les personnes qui ne l'ont pas encore fait.

In [ ]:
ctx = m[m["expose_contextual"] == True].copy()
ctx["groupe"] = np.where(ctx["expose_tv"] == True, "Exposés TV", "Non exposés")
print(ctx["groupe"].value_counts())

# Contrôle : les deux groupes reçoivent-ils le contextuel aux mêmes moments ?
ctx["contextuel_apres"] = ctx["premiere_contextual"] >= debut_tv
print("\n1re pub contextuelle après le début de la TV (%) :")
print((ctx.groupby("groupe")["contextuel_apres"].mean() * 100).round(1))

# 1. CA par personne et par jour (non-visiteurs compris)
did_tv(ctx, "Exposés au contextuel, visiteurs ou non")

# 2. Nouveaux visiteurs et nouveaux acheteurs
def nouveaux(df, col, nom):
    "Taux de 1re visite (ou de 1er achat) par jour, parmi ceux qui ne l'ont pas encore fait."
    taux = {}
    for gr, d in df.groupby("groupe"):
        k_avant = (d[col] < debut_tv).sum()
        k_apres = (d[col] >= debut_tv).sum()
        taux[gr] = {
            "k_avant": k_avant, "k_apres": k_apres,
            "Taux avant (pour 1 000 / jour)": k_avant / (len(d) * jours_avant) * 1000,
            "Taux après (pour 1 000 / jour)": k_apres / ((len(d) - k_avant) * jours_apres) * 1000,
        }
    t = pd.DataFrame(taux).T
    t["Évolution (x)"] = t["Taux après (pour 1 000 / jour)"] / t["Taux avant (pour 1 000 / jour)"]
    rapport = t.loc["Exposés TV", "Évolution (x)"] / t.loc["Non exposés", "Évolution (x)"]
    se = np.sqrt((1 / t[["k_avant", "k_apres"]].astype(float)).to_numpy().sum())
    print(f"\n{nom} : évolution des exposés TV / évolution des non exposés = {rapport:.2f} "
          f"[IC 95 % : {rapport * np.exp(-1.96 * se):.2f} ; {rapport * np.exp(1.96 * se):.2f}]")
    display(t.drop(columns=["k_avant", "k_apres"]).round(3))

nouveaux(ctx, "premiere_visite", "Nouveaux visiteurs")
nouveaux(ctx, "premiere_commande", "Nouveaux acheteurs")

In [ ]:
groupe_ctx = pd.Series(ctx["groupe"].to_numpy(), index=ctx["customer_id"])
taille_ctx = groupe_ctx.value_counts()

# Écart de CA par personne et par semaine
cmd_sem = parcours.loc[parcours["type"] == "Order", ["customer_id", "timestamp_utc", "CA"]]
cmd_sem["groupe"] = cmd_sem["customer_id"].map(groupe_ctx)
cmd_sem = cmd_sem.dropna(subset=["groupe"])
cmd_sem["semaine"] = cmd_sem["timestamp_utc"].dt.to_period("W").dt.start_time
ca_sem = cmd_sem.groupby(["semaine", "groupe"])["CA"].sum().unstack() / taille_ctx

# Nouveaux visiteurs par semaine, pour 1 000 personnes
nv = ctx.dropna(subset=["premiere_visite"])
nv_sem = (nv.groupby([nv["premiere_visite"].dt.to_period("W").dt.start_time, "groupe"]).size()
            .unstack() / taille_ctx * 1000)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
(ca_sem["Exposés TV"] - ca_sem["Non exposés"]).plot(ax=axes[0], marker="o", color="black")
axes[0].axhline(0, color="grey", lw=0.8)
axes[0].set_title("Écart de CA par personne et par semaine (exposés TV - non exposés)"); axes[0].set_ylabel("$")
nv_sem.plot(ax=axes[1], marker="o")
axes[1].set_title("Nouveaux visiteurs pour 1 000 personnes, par semaine")
for ax in axes:
    ax.axvline(debut_tv, color="red", ls="--", label="Début TV")
    ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

**Récapitulatif :**

- Population : 3,3 M de personnes exposées au contextuel, dont 82 000 exposées à la TV. Les deux groupes reçoivent le contextuel à des moments proches (première publicité contextuelle après le 29 février : environ 37 % contre 42 %). Avant la TV, les exposés TV achetaient déjà 2,5 fois plus souvent (1,5 % d'acheteurs contre 0,6 %) : on compare des évolutions, pas des niveaux.
- CA : un effet très faible, de l'ordre de 0,001 \$ par personne et par jour, sans rupture visible au lancement de la TV : l'écart hebdomadaire entre les deux groupes reste dans la même fourchette avant et après le 29 février.
- Nouveaux visiteurs : le taux de premières visites baisse un peu moins chez les exposés TV après le 29 février (rapport de 1,06 ; IC 95 % : 1,03 à 1,10). La TV amènerait environ 6 % de nouveaux visiteurs en plus.
- Nouveaux acheteurs : ce surplus de visiteurs ne se traduit pas en premiers achats (rapport de 0,93 ; IC 95 % : 0,86 à 1,01).


**Bilan sur la TV :** selon la population étudiée, l'effet mesuré sur le CA va d'environ −4 % (clients déjà présents) à +14 % (tous les visiteurs, effet porté par les nouveaux visiteurs).
La TV semble entraîner une légère hausse des premières visites, sans augmentation du nombre de nouveaux acheteurs. Si la TV a un effet sur les ventes, il est faible. L'estimation de 3 \$ de CA par dollar dépensé (418 000 \$ pour 135 000 \$ de budget, 3.4.1) n'est pas fiable : elle est notamment portée par les nouveaux visiteurs et ne se retrouve pas chez les clients déjà présents. De plus, le chiffre d'affaires ne tient pas compte des coûts et ne permet donc pas de mesurer la rentabilité de la campagne.

### Synthèse : la publicité fait-elle vendre ?

Les constats portent sur les publicités rattachées à une personne (`pop`, `parcours`) et, pour le budget, sur toutes les impressions (niveau B), du 1er janvier au 30 juin 2024.

**Réponse :** on ne peut pas montrer que la publicité fait vendre. Les visiteurs retargetés achètent plus, mais parce qu'ils étaient déjà intéressés avant de recevoir la publicité. Pour la TV, la hausse vient de nouveaux visiteurs dont une partie est venue sur le site avant d'avoir vu la publicité. Cela ne veut pas dire que la publicité est inutile. On n’a pas trouvé de preuve que la pub marche. Mais on n’a pas non plus prouvé qu’elle ne marche pas : elle a peut-être un petit effet, qu’on ne peut pas voir parce que les exposés et les non-exposés ne sont pas comparables.

| Constat | Chiffres clés | Section |
|---|---|---|
| **Budget** : la TV, lancée le 29 février, prend la majorité du budget ; le retargeting coûte près de 3 fois plus cher à l'impression que le contextuel | TV 135 K\$ (59 %, CPM 23,3 \$) ; contextuel 51 K\$ (CPM 3,7 \$) ; retargeting 42 K\$ (CPM d'environ 11 \$) | 3.1.1 |
| **Comparabilité** : `mapping` ne contient que des personnes exposées ou venues sur le site ; les comparaisons se font entre visiteurs | « Aucune pub » : 100 % de visiteurs ; 1,17 M de visiteurs comparés | 3.2 |
| **Contextuel** : un public qui achète peu, sans signe d'effet positif | 11 % d'acheteurs contre 42 % sans pub ; ratio de 0,26 à 0,28, que la pub précède ou suive la première visite | 3.3 |
| **Retargeting** : un écart positif dû au ciblage de visiteurs déjà actifs | Les visiteurs retargetés achètent plus (48 % d'acheteurs contre 42 % sans pub). Ceux qui ont reçu la pub avant leur visite achètent comme les visiteurs sans pub (0,98) | 3.2, 3.3 |
| **TV** : pas d'effet robuste sur les ventes ; au mieux un léger effet sur la venue sur le site | Effet sur le CA de −4 % à +14 % selon la population ; +6 % de premières visites, pas plus de nouveaux acheteurs (0,93) | 3.3.2, 3.4 |
| **Profil** : l'âge et le revenu n'expliquent aucun des écarts | Ratios standardisés égaux aux ratios bruts à 0,01 près | 3.3.1 |

**Limites :**

- `mapping` ne contient pas de personnes à la fois non exposées et jamais venues sur le site : l'effet de la publicité sur la venue n'est mesurable que pour la TV, à travers les exposés au contextuel (3.4.2) ;
- six mois d'historique seulement : le retargeting cible dès le 1er janvier des personnes sans visite observée (car venues avant), et « première visite » signifie première visite observée ;
- une mesure incomplète : 13 % des impressions programmatiques et 5 % des impressions TV ne sont rattachées à personne.